# Suite FRM — The framework

Every dialect implements the same protocols (`mbse.Expressions.Framework`): its terms are values of the framework's
`Term`, the union of the dialects' `Term`s, stored through its install, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms` reads any dialect's terms by its kinds (`Terms.Kind`, `Terms.Dialect`); this suite
checks that machinery on every dialect at once, and on an extension of Patterns declared here. The dialects' own rules are
suite DIA, and translation suite TRN.

In [ ]:
import functools
import math

from mbse.Expressions import Framework, Translators
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Cpp import Expressions as Cpp
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.LaTeX import Expressions as Latex
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Python3
from mbse.Expressions.Arm import AAPCS64
from mbse.Expressions.SystemV import AMD64
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.C import Preprocessor as CPreprocessor, Types as CTypes  # the retrofit's new dialects
from mbse.Expressions.Cpp import RegExp as CppRegExp, Types as CppTypes  # and the regular expressions std::regex reads
from mbse.Expressions.POSIX import BRE as PosixBRE, ERE as PosixERE  # and POSIX's, which a C++ regex may hold
from mbse.Expressions.Python import Patterns as PythonPatterns, RegExp as PythonRegExp, Types as PythonTypes  # the retrofit's, built
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, RegExp as TypeScriptRegExp, Types as TypeScriptTypes
from mbse.Expressions.Patterns.Expressions import Domains as BD, Simplifiers as V, Expressions as E
from mbse.Expressions.C.Expressions import Simplifiers as CV, Expressions as C
from mbse.Expressions.Cpp.Expressions import Simplifiers as CPV, Expressions as CPP
from mbse.Expressions.Excel.Expressions import Simplifiers as XV, Expressions as X
from mbse.Expressions.LaTeX.Expressions import Expressions as L
from mbse.Expressions.MATLAB.Expressions import Simplifiers as MV, Expressions as M
from mbse.Expressions.Python.Python3 import Simplifiers as PV, Expressions as P
from mbse.Expressions.Python.Types import Expressions as PYT
from mbse.Expressions.Python.Patterns import Expressions as PYP
from mbse.Expressions.Python.RegExp import Expressions as PYR
from mbse.Expressions.SystemVerilog.Expressions import Simplifiers as SVV, Expressions as SV
from mbse.Expressions.SystemVerilog.Types import Expressions as SVT
from mbse.Expressions.SystemVerilog.Patterns import Expressions as SVP
from mbse.Expressions.SystemVerilog.Assertions import Expressions as SVA
from mbse.Expressions.SystemVerilog.Constraints import Expressions as SVC
from mbse.Expressions.SystemVerilog.Preprocessor import Expressions as SVPP
from mbse.Expressions.TypeScript.TypeScript5 import Simplifiers as TV, Expressions as TS
from mbse.Expressions.TypeScript.Types import Expressions as TST
from mbse.Expressions.TypeScript.Patterns import Expressions as TSP
from mbse.Expressions.TypeScript.RegExp import Expressions as TSR
from mbse.Expressions.POSIX.BRE import Expressions as PXB
from mbse.Expressions.POSIX.ERE import Expressions as PXE
from mbse.Expressions.Framework import Domains as D, Errors as FE, Simplifiers as FV, Symbolics as SY, \
    Terms as F, Translators as FT
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

RETROFIT = [PythonTypes, PythonPatterns, TypeScriptTypes, TypeScriptPatterns, CTypes, CppTypes, CPreprocessor,  # installed so
            PythonRegExp, TypeScriptRegExp, CppRegExp, PosixBRE, PosixERE]  # the regular expression dialects last, POSIX's
# after C++'s
store = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,  # the standard
                Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install,  # dialects first
                C.install, Cpp.install, AMD64.install, AAPCS64.install, SystemVerilog.install, SVTypes.install,
                SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install, TypeScript5.install,
                *(package.install for package in RETROFIT)]:  # each after its family's dialect
    store.transaction(install)
store.transaction(Patterns.define)  # Patterns' operations, which its validation reads
for define in [C.define, Cpp.define, AMD64.define, AAPCS64.define]:  # C's, C++'s and the ABIs' types, which their
    store.transaction(define)  # evaluation, inference and validation read through a term's holder
store.transaction(SystemVerilog.define)  # SystemVerilog's types, which its evaluation, inference and validation read
for define in [SVTypes.define, SVPatterns.define, SVAssertions.define, SVConstraints.define, SVPreprocessor.define]:
    store.transaction(define)  # the editions SystemVerilog's sublanguages cover
for package in RETROFIT:
    store.transaction(package.define)  # the retrofit's new dialects' data: the editions they cover, or none yet
Term = store.registered("Expressions.Framework.Term")
AMD64_ = store.Instances.Expressions.SystemV.AMD64  # the ABI a C or C++ term's holder names


def term(spec, dialect=E.DIALECT):
    """The term value `spec` writes, the root of a holder written in a transaction; a C or C++ term's holder names the
    ABI its terms are read under, x86-64's System V ABI."""
    with store.transaction():
        if dialect.tag in ("c", "cpp"):
            return F.term(store.Expressions.Framework.Expression(lambda e: e.root(dialect.spec(spec)).abi([AMD64_])))
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(dialect.spec(spec))))


person = E.variable("person")
# TypeScript5 has no writers: its terms are written through the builders
ts_person = lambda u: u.identifier(lambda r: r.name("person"))  # noqa: E731: TypeScript5's `ts_person`, in any union that has identifiers
typescript5 = lambda u: u.sequenceexpression(lambda s: s.expressions([  # noqa: E731: every kind, written through the builders
    lambda e: e.parenthesizedexpression(lambda r: r.expression(lambda x: x.assignmentexpression(lambda a: a  # ({ a: [ts_person, , ...ts_person], d: { ts_person }, ts_person = 1 } = ts_person),
        # a destructuring assignment's targets: references to what it assigns
        .left(lambda t: t.objectpattern(lambda o: o.properties([
            lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("a"))).value(
                lambda v: v.arraypattern(lambda ap: ap.elements([ts_person, lambda y: y.elision(lambda h: h),
                                                                  lambda y: y.restelement(lambda z: z.argument(ts_person))])))),
            lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("d"))).value(
                lambda v: v.objectpattern(lambda op: op.properties([lambda y: y.property(lambda z: z.propertyKind("init").key(ts_person).shorthand(True).value(ts_person))])))),
            lambda p: p.property(lambda q: q.propertyKind("init").key(ts_person).shorthand(True).value(
                lambda v: v.assignmentpattern(lambda ap: ap.left(ts_person).right(
                    lambda g: g.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)))))))])))
        .operator("=").right(ts_person)))),
    lambda e: e.functionexpression(lambda r: r.isAsync(True).generator(True).id(lambda i: i.name("g")).typeParameters(  # async function* g<T>(x: T = this.y) {}
        lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([lambda p: p.assignmentpattern(lambda ap: ap
            .left(lambda g: g.identifier(lambda i: i.name("x").typeAnnotation(lambda ta: ta.typeAnnotation(
                lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("T"))))))))
            .right(lambda g: g.memberexpression(lambda m: m.object(lambda o: o.thisexpression(lambda h: h)).property(lambda y: y.identifier(lambda i: i.name("y"))))))])),
    lambda e: e.arrowfunctionexpression(lambda r: r.isAsync(True).typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("U"))]))  # async <U>(p: U, ...r: U[]) => await import.meta.resolve(p)
        .params([lambda p: p.identifier(lambda i: i.name("p").typeAnnotation(lambda ta: ta.typeAnnotation(
                     lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("U"))))))),
                 lambda p: p.restelement(lambda z: z.argument(lambda g: g.identifier(lambda i: i.name("r"))).typeAnnotation(lambda ta: ta.typeAnnotation(
                     lambda t: t.tsarraytype(lambda at: at.elementType(lambda y: y.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("U")))))))))])
        .body(lambda b: b.awaitexpression(lambda w: w.argument(lambda c: c.callexpression(lambda ce: ce.callee(lambda f: f.memberexpression(lambda m: m
            .object(lambda o: o.metaproperty(lambda mp: mp.meta(lambda i: i.name("import")).property(lambda i: i.name("meta"))))
            .property(lambda y: y.identifier(lambda i: i.name("resolve"))))).arguments([lambda y: y.identifier(lambda i: i.name("p"))])))))),
    lambda e: e.classexpression(lambda r: r.decorators([lambda d: d.expression(ts_person)]).id(lambda i: i.name("K"))  # @ts_person class K<T> extends ts_person.Base<T> implements ts_person {}
        .typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))]))
        .superClass(lambda c: c.memberexpression(lambda m: m.object(ts_person).property(lambda y: y.identifier(lambda i: i.name("Base")))))
        .superTypeArguments(lambda a: a.params([lambda t: t.tstypereference(lambda tr: tr.typeName(lambda n: n.identifier(lambda j: j.name("T"))))]))
        .implements([lambda c: c.expression(ts_person)])),
    lambda e: e.newexpression(lambda r: r.callee(lambda c: c.memberexpression(lambda m: m.object(ts_person).property(  # new ts_person.C<number>(...ts_person)
        lambda y: y.identifier(lambda i: i.name("C"))))).typeArguments(lambda a: a.params([lambda t: t.tsnumberkeyword(lambda k: k)]))
        .arguments([lambda y: y.spreadelement(lambda z: z.argument(ts_person))])),
    lambda e: e.taggedtemplateexpression(lambda r: r.tag(ts_person).quasi(lambda q: q.quasis([  # ts_person`a${ts_person}b`
        lambda y: y.raw("a"), lambda y: y.raw("b").tail(True)]).expressions([ts_person]))),
    lambda e: e.templateliteral(lambda r: r.quasis([lambda y: y.raw("x"), lambda y: y.raw("").tail(True)]).expressions([ts_person])),  # `x${ts_person}`
    lambda e: e.updateexpression(lambda r: r.operator("++").argument(ts_person)),  # ts_person++
    lambda e: e.unaryexpression(lambda r: r.operator("typeof").argument(ts_person)),  # typeof ts_person
    lambda e: e.binaryexpression(lambda r: r.left(lambda y: y.privateidentifier(lambda i: i.name("p"))).operator("in").right(ts_person)),  # #p in ts_person
    lambda e: e.logicalexpression(lambda r: r.left(ts_person).operator("??").right(lambda y: y.constant(lambda c: c.singleton("null")))),  # ts_person ?? null
    lambda e: e.conditionalexpression(lambda r: r.test(ts_person).consequent(  # ts_person ? /x/g : 1n
        lambda y: y.constant(lambda c: c.pattern("x").flags("g"))).alternate(lambda y: y.literal(lambda li: li.value(lambda n: n.as_scalar(1))))),
    lambda e: e.yieldexpression(lambda r: r.delegate(True).argument(ts_person)),  # yield* ts_person
    lambda e: e.tsasexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.tsstringkeyword(lambda k: k))),  # ts_person as string
    lambda e: e.tssatisfiesexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.tsobjectkeyword(lambda k: k))),  # ts_person satisfies object
    lambda e: e.tstypeassertion(lambda r: r.typeAnnotation(lambda t: t.tsanykeyword(lambda k: k)).expression(ts_person)),  # <any>ts_person
    lambda e: e.tsnonnullexpression(lambda r: r.expression(ts_person)),  # ts_person!
    lambda e: e.tsinstantiationexpression(lambda r: r.expression(ts_person).typeArguments(lambda a: a.params([lambda t: t.tsstringkeyword(lambda k: k)]))),  # ts_person<string>
    lambda e: e.arrayexpression(lambda r: r.elements([  # [1, , ...ts_person]: the framework's list, its elements any dialect's terms
        lambda y: y.typescript5(lambda t: t.literal(lambda li: li.value(lambda n: n.as_scalar(1.0)))), lambda y: y.typescript5(lambda t: t.elision(lambda h: h)),
        lambda y: y.typescript5(lambda t: t.spreadelement(lambda z: z.argument(ts_person)))])),
    lambda e: e.objectexpression(lambda r: r.properties([  # { m(): void, ...ts_person, [ts_person]: "s" }
        lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("m"))).method(True).value(
            lambda v: v.tsemptybodyfunctionexpression(lambda f: f.returnType(lambda a: a.typeAnnotation(lambda t: t.tsvoidkeyword(lambda k: k)))))),
        lambda p: p.spreadelement(lambda z: z.argument(ts_person)),
        lambda p: p.property(lambda q: q.propertyKind("init").computed(True).key(ts_person).value(lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar("s")))))])),
    lambda e: e.jsxelement(lambda r: r  # <div id="x" {...ts_person}>{}{ts_person}<>text</></div>
        .openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i.name("div"))).attributes([
            lambda a: a.jsxattribute(lambda t: t.name(lambda n: n.jsxidentifier(lambda i: i.name("id"))).value(
                lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar("x")).text('"x"')))),
            lambda a: a.jsxspreadattribute(lambda z: z.argument(ts_person))]))
        .children([lambda c: c.jsxexpressioncontainer(lambda x: x.expression(lambda y: y.jsxemptyexpression(lambda h: h))),
                   lambda c: c.jsxexpressioncontainer(lambda x: x.expression(ts_person)),
                   lambda c: c.jsxfragment(lambda f: f.openingFragment(lambda h: h).children([lambda t: t.jsxtext(lambda j: j.raw("text"))]).closingFragment(lambda h: h))])
        .closingElement(lambda c: c.name(lambda n: n.jsxidentifier(lambda i: i.name("div"))))),
    lambda e: e.jsxfragment(lambda f: f.openingFragment(lambda h: h).children([lambda c: c.jsxelement(lambda x: x  # <><br /></>
        .openingElement(lambda o: o.name(lambda n: n.jsxidentifier(lambda i: i.name("br"))).selfClosing(True)))]).closingFragment(lambda h: h)),
    lambda e: e.chainexpression(lambda r: r.expression(lambda y: y.memberexpression(lambda m: m.object(ts_person).optional(True).property(  # ts_person?.a
        lambda z: z.identifier(lambda i: i.name("a")))))),
    lambda e: e.memberexpression(lambda m: m.object(lambda o: o.super(lambda h: h)).property(lambda y: y.identifier(lambda i: i.name("x")))),  # super.x
    lambda e: e.importexpression(lambda r: r.source(ts_person)),  # import(ts_person)
]))
c_person, cpp_person = C.identifier("person"), CPP.identifier("person")
c_int = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("int"))]))  # noqa: E731: `int`, in C's or C++'s
uint8 = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda n: n.name(  # noqa: E731: `uint8_t`, in C's or C++'s
    lambda m: m.identifier(lambda d: d.spelling("uint8_t"))))]))
c_every = lambda u: u.initializerlist(lambda r: r.items([  # noqa: E731: every kind of C's, written through the builders and its writers
    C.conditional(C.binary("&&", C.unary("!", C.member(c_person, "a")), C.binary(">", C.subscript(C.member(c_person, "b", "->"), 0), C.literal(1.5, "f"))),
                  C.cast(uint8, C.call("person", 1)), C.literal(5, "u")),  # the kinds C evaluates
    lambda i: i.constant(lambda c: c.singleton("nullptr")),  # nullptr
    lambda i: i.concatenatedstring(lambda c: c.parts([C.literal("a"), C.literal("b")])),  # "a" "b"
    lambda i: i.parenthesizedexpression(lambda p: p.expression(c_person)),  # (person)
    lambda i: i.genericselection(lambda g: g.controlling(c_person).associations([  # _Generic(person, int: 1, default: 0)
        lambda a: a.type(c_int).value(C.literal(1)), lambda a: a.value(C.literal(0))])),
    lambda i: i.postfixexpression(lambda p: p.operand(c_person).operator("++")),  # person++
    lambda i: i.sizeofexpression(lambda s: s.operand(c_int)),  # sizeof(int): a type name where an expression may stand
    lambda i: i.alignofexpression(lambda s: s.keyword("_Alignof").operand(c_person)),  # _Alignof(person)
    lambda i: i.assignmentexpression(lambda a: a.left(c_person).operator("+=").right(C.literal(1))),  # person += 1
    lambda i: i.designatedinitializer(lambda d: d.designators([  # .x[0] = person
        lambda x: x.fielddesignator(lambda f: f.name(lambda j: j.spelling("x"))), lambda x: x.indexdesignator(lambda f: f.index(C.literal(0)))]).initializer(
        lambda x: x.equalinitializer(lambda e: e.value(c_person)))),
    lambda i: i.compoundliteralexpression(lambda c: c.type(lambda t: t.typeid(lambda y: y.specifiers([  # (int[]){1}
        lambda s: s.primitivetypespecifier(lambda p: p.keyword("int"))]).declarator(lambda d: d.arraydeclarator(lambda a: a)))).initializer(
        lambda li: li.items([C.literal(1)]))),
    lambda i: i.statementexpression(lambda s: s),  # ({}): GNU's, its body of statements not part of the term
    lambda i: i.extensionexpression(lambda e: e.operand(c_person)),  # __extension__ person
]))
cpp_name = lambda spelling: lambda n: n.identifier(lambda i: i.spelling(spelling))  # noqa: E731: a C++ name that is an identifier
cpp_every = lambda u: u.initializerlist(lambda r: r.items([  # noqa: E731: every kind of C++'s, written through the builders and its writers
    CPP.conditional(CPP.binary("&&", CPP.unary("!", CPP.member(cpp_person, "a")), CPP.binary(">", CPP.subscript(CPP.member(cpp_person, "b", "->"), 0), CPP.literal(1.5, "f"))),
                    CPP.cast(uint8, CPP.call("person", 1)), CPP.literal(5, "u")),  # the kinds C++ evaluates
    lambda i: i.constant(lambda c: c.singleton("nullptr")),  # nullptr
    lambda i: i.userdefinedliteral(lambda d: d.literal(lambda li: li.value(lambda v: v.as_scalar(10))).suffix("_km")),  # 10_km
    lambda i: i.concatenatedstring(lambda c: c.parts([CPP.literal("a"), lambda x: x.literal(  # "a" R"(b)"
        lambda li: li.value(lambda v: v.as_scalar("b")).raw(True))])),
    lambda i: i.idexpression(lambda d: d.name(lambda n: n.qualifiedname(lambda q: q.qualifiers([cpp_name("std")]).name(cpp_name("cout"))))),  # std::cout
    lambda i: i.thisexpression(lambda t: t),  # this
    lambda i: i.parenthesizedexpression(lambda x: x.expression(cpp_person)),  # (person)
    lambda i: i.lambdaexpression(lambda la: la.captures([  # [=, &person](auto y) {}: its body of statements not part of the term
        lambda c: c.defaultcapture(lambda d: d.mode("=")), lambda c: c.simplecapture(lambda s: s.by_reference(True).name(lambda j: j.spelling("person")))]).declarator(
        lambda d: d.parameters([lambda q: q.parameterdeclaration(lambda pd: pd.specifiers([lambda s: s.placeholdertypespecifier(lambda h: h)]).declarator(
            lambda dd: dd.iddeclarator(lambda i2: i2.name(cpp_name("y")))))]))),
    lambda i: i.foldexpression(lambda f: f.left(cpp_person).operator("+")),  # (person + ...)
    lambda i: i.requiresexpression(lambda q: q.parameters([lambda a: a.parameterdeclaration(  # requires (int z) { z + 1; }
        lambda pd: pd.specifiers([lambda s: s.primitivetypespecifier(lambda k: k.keyword("int"))]).declarator(
            lambda dd: dd.iddeclarator(lambda i2: i2.name(cpp_name("z")))))]).requirements([
        lambda r2: r2.simplerequirement(lambda s: s.expression(CPP.binary("+", CPP.identifier("z"), 1)))])),
    lambda i: i.functionalcastexpression(lambda f: f.type(lambda s: s.primitivetypespecifier(lambda q: q.keyword("int"))).initializer(  # int(person)
        lambda x: x.parenthesizedinitializer(lambda pi: pi.arguments([cpp_person])))),
    lambda i: i.postfixexpression(lambda x: x.operand(cpp_person).operator("--")),  # person--
    lambda i: i.namedcastexpression(lambda n: n.operator("static_cast").type(c_int).operand(cpp_person)),  # static_cast<int>(person)
    lambda i: i.typeidexpression(lambda t: t.operand(cpp_person)),  # typeid(person)
    lambda i: i.awaitexpression(lambda a: a.operand(cpp_person)),  # co_await person
    lambda i: i.sizeofexpression(lambda s: s.operand(c_int)),  # sizeof(int)
    lambda i: i.sizeofpackexpression(lambda s: s.pack(lambda j: j.spelling("Ts"))),  # sizeof...(Ts)
    lambda i: i.alignofexpression(lambda s: s.keyword("alignof").operand(c_int)),  # alignof(int)
    lambda i: i.noexceptexpression(lambda s: s.operand(cpp_person)),  # noexcept(person)
    lambda i: i.newexpression(lambda n: n.type(c_int).initializer(lambda x: x.initializerlist(lambda il: il.items([CPP.literal(1)])))),  # new int{1}
    lambda i: i.deleteexpression(lambda d: d.array(True).operand(cpp_person)),  # delete[] person
    lambda i: i.assignmentexpression(lambda a: a.left(cpp_person).operator("=").right(CPP.literal(0))),  # person = 0
    lambda i: i.throwexpression(lambda t: t),  # throw
    lambda i: i.yieldexpression(lambda y: y.operand(cpp_person)),  # co_yield person
    lambda i: i.packexpansion(lambda x: x.pattern(cpp_person)),  # person...
    lambda i: i.designatedinitializer(lambda d: d.designators([lambda x: x.fielddesignator(lambda f: f.name(lambda j: j.spelling("x")))]).initializer(  # .x = person
        lambda x: x.equalinitializer(lambda e: e.value(cpp_person)))),
    lambda i: i.statementexpression(lambda s: s),  # ({})
    lambda i: i.extensionexpression(lambda e: e.operand(cpp_person)),  # __extension__ person
]))
CT, XT, CPRE = CTypes.Expressions.DIALECT, CppTypes.Expressions.DIALECT, CPreprocessor.Expressions.DIALECT
XR = CppRegExp.Expressions.DIALECT
C_FAMILY = [CT, XT, CPRE, XR]  # the C family's new dialects, built: C's and C++'s types, the preprocessor both read, and
# C++'s regular expressions
POSIX = [PXB.DIALECT, PXE.DIALECT]  # POSIX's regular expressions, which a C++ regex naming a POSIX grammar holds
c_spelled = lambda spelling: lambda i: i.spelling(spelling)  # noqa: E731: an identifier, C's or C++'s


def enumeration(types, family):
    """Every kind of a Types dialect: an enumeration's type name, its body's enumerators, and a directive and a comment
    among them, C Preprocessor's: `enum E { A = 1, #ifdef B_ B, #endif //last }`."""
    return lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.enumspecifier(lambda e: e.key("enum").name(
        lambda n: n.identifier(c_spelled("E"))).body(True).enumerators([
            lambda n: n.enumerator(lambda r: r.name(c_spelled("A")).value(family.literal(1))),
            lambda n: n.ifdefdirective(lambda d: d.name(c_spelled("B_")).items([types.spec(lambda m: m.enumerator(
                lambda r: r.name(c_spelled("B"))))])),
            lambda n: n.comment(lambda c: c.text("last").trailing(True))]).trailing_comma(True))]))


c_preprocessor_every = lambda u: u.ifdirective(lambda d: d.condition(  # noqa: E731: every kind of C Preprocessor's
    C.binary("&&", C.DIALECT.spec(lambda v: v.definedexpression(lambda x: x.name(c_spelled("X")))), C.binary(  # a C condition
        ">", C.identifier("X"), 1))).items([CT.spec(lambda m: m.enumerator(lambda r: r.name(c_spelled("A")))),  # its items
            CPRE.spec(lambda m: m.includedirective(lambda i: i.path("a.h").system(True))),
            CPRE.spec(lambda m: m.comment(lambda c: c.block(True).text(" one ")))]).alternative(
    lambda a: a.elifdirective(lambda e: e.condition(CPP.binary("==", CPP.identifier("Y"), 2)).alternative(  # a C++ one
        lambda b: b.elifdefdirective(lambda f: f.negated(True).name(c_spelled("Z")).alternative(
            lambda c: c.elsedirective(lambda g: g.items([
                CPRE.spec(lambda m: m.definedirective(lambda h: h.name(c_spelled("F")).function_like(True).parameters([
                    c_spelled("a")]).variadic(True).replacement("a + 1"))),
                CPRE.spec(lambda m: m.otherdirective(lambda o: o.directive("pragma").text("once"))),
                CPRE.spec(lambda m: m.ifdefdirective(lambda o: o.name(c_spelled("W"))))]))))))))
basic = E.let_("a", person.age, E.variable("a").ge(18).and_(person.has("email")))
# SystemVerilog's: every kind of its Expressions and its helpers, with its Types and the Preprocessor's macro usage
# written through the builders where a writer has none
sv_person = SV.identifier("person")
sv_identifier = lambda spelling: lambda i: i.spelling(spelling)  # noqa: E731: an identifier, of its spelling
sv_member = lambda u: u.memberexpression(lambda m: m.value(sv_person).member(sv_identifier("a")))  # noqa: E731: person.a, in any union
sv_atom = lambda keyword: lambda t: t.integeratomtype(lambda a: a.keyword(keyword))  # noqa: E731: `int`, a Types term
sv_every = SV.binary("->", SV.inside(SV.member(sv_person, "a"), 1, SV.span(2, 5)), SV.concatenation(  # noqa: E731
    lambda u: u.conditionalexpression(lambda c: c.condition(lambda m: m.matchesexpression(lambda x: x.value(sv_person).pattern(SV.literal(3))))
        .attributes([lambda a: a.specs([lambda s: s.name(sv_identifier("keep")).value(SV.literal(1))])])
        .consequence(lambda q: q.literal(lambda r: r.value(lambda v: v.as_scalar(31)).base("h"))).alternative(SV.vector("10x1", True))),
    SV.select(sv_person, 0), SV.range_(sv_person, 7, 4), SV.unary("-", sv_person),
    lambda u: u.incrementexpression(lambda i: i.operator("++").postfix(True).operand(sv_person)),
    lambda u: u.parenthesizedexpression(lambda p: p.expression(lambda e: e.assignmentexpression(lambda a: a.target(sv_person).operator("+=").value(SV.literal(1))))),
    SV.replication(2, sv_person),
    lambda u: u.assignmentpattern(lambda a: a.type(sv_atom("int")).items([lambda i: i.patternitem(lambda p: p.key(sv_person).value(SV.literal(1))),
                                                                          lambda i: i.patternitem(lambda p: p.value(SV.literal(0)))])),
    lambda u: u.assignmentpattern(lambda a: a.count(SV.literal(2)).items([sv_person])),
    lambda u: u.thisexpression(lambda t: t), lambda u: u.superexpression(lambda t: t),
    lambda u: u.newexpression(lambda n: n.scope(lambda s: s.scopedname(lambda c: c.scope(lambda x: x.identifier(sv_identifier("pkg"))).name(
        lambda x: x.parameterizedname(lambda q: q.name(sv_identifier("c")).parameters([SV.literal(8)]))))).arguments([
        SV.literal(1), lambda a: a.namedconnection(lambda c: c.name(sv_identifier("n")).value(sv_person)), lambda a: a.emptyargument(lambda e: e)])),
    lambda u: u.newexpression(lambda n: n.scope(lambda s: s.superexpression(lambda x: x)).defaulted(True)),
    lambda u: u.newcopyexpression(lambda n: n.value(sv_person)), lambda u: u.newarrayexpression(lambda n: n.size(SV.literal(4)).value(sv_person)),
    lambda u: u.randomizewithexpression(lambda r: r.call(SV.method(sv_person, "randomize")).restricted(True).variables([sv_identifier("a")]).items(
        [SV.binary("<", sv_person, 5)])),
    SV.iterate(sv_person, "sum", "q", SV.binary(">", SV.identifier("q"), 0)),
    lambda u: u.callexpression(lambda c: c.callee(sv_person).attributes([lambda a: a.specs([lambda s: s.name(sv_identifier("pure"))])]).arguments([
        sv_person, lambda a: a.namedconnection(lambda n: n.name(sv_identifier("n")).implicit(True)), lambda a: a.emptyargument(lambda e: e)])),
    SV.call("$bits", sv_atom("int")), SV.call("$clog2", sv_person), SV.cast(sv_atom("byte"), sv_person), SV.cast(8, sv_person),
    lambda u: u.dollarexpression(lambda d: d), lambda u: u.rootexpression(lambda r: r), lambda u: u.emptyqueue(lambda e: e),
    lambda u: u.mintypmaxexpression(lambda m: m.min(SV.literal(1)).typ(SV.literal(2)).max(SV.literal(3))),
    lambda u: u.taggedexpression(lambda t: t.member(sv_identifier("Valid")).value(sv_person)),
    lambda u: u.predicateexpression(lambda p: p.conditions([lambda c: c.matchesexpression(lambda m: m.value(sv_person).pattern(SV.literal(1))), sv_person])),
    lambda u: u.streamingconcatenation(lambda s: s.operator("<<").slice(sv_atom("byte")).items([
        lambda i: i.expression(sv_person).left(SV.literal(0)).operator(":").right(SV.literal(1))])),
    lambda u: u.nameexpression(lambda n: n.name(lambda x: x.localname(lambda l: l.name(sv_identifier("y"))))),
    lambda u: u.nameexpression(lambda n: n.name(lambda x: x.scopedname(lambda c: c.scope(lambda y: y.unitname(lambda z: z)).name(lambda y: y.identifier(sv_identifier("z")))))),
    lambda u: u.nameexpression(lambda n: n.name(lambda x: x.interfacetypename(lambda i: i.interface(sv_identifier("bus")).name(sv_identifier("t"))))),
    lambda u: u.nameexpression(lambda n: n.name(lambda x: x.interfacemethodname(lambda i: i.port(sv_identifier("port")).name(sv_identifier("m"))))),
    lambda u: u.macrousage(lambda m: m.name(sv_identifier("WIDTH"))), lambda u: u.constant(lambda c: c.fill("1")),
    lambda u: u.constant(lambda c: c.singleton("null")), SV.literal(1.5)))
types_every = lambda u: u.structtype(lambda s: s.keyword("union").qualifier("tagged").packed(True).signing("signed").members([  # noqa: E731
    lambda m: m.random("rand").type(lambda t: t.integervectortype(lambda v: v.keyword("logic").signing("signed").dimensions([
        lambda d: d.rangedimension(lambda r: r.left(SV.literal(7)).right(SV.literal(0))), lambda d: d.sizedimension(lambda z: z.size(SV.literal(4))),
        lambda d: d.unsizeddimension(lambda z: z), lambda d: d.associativedimension(lambda a: a.type(sv_atom("int"))),
        lambda d: d.associativedimension(lambda a: a), lambda d: d.queuedimension(lambda q: q),
        lambda d: d.queuedimension(lambda q: q.bound(SV.literal(8)))]))).declarators([lambda d: d.name(sv_identifier("a"))]),
    lambda m: m.attributes([lambda a: a.specs([lambda s: s.name(sv_identifier("keep"))])]).type(sv_atom("int")).declarators([
        lambda d: d.name(sv_identifier("b")).dimensions([lambda x: x.unsizeddimension(lambda y: y)]).value(SV.literal(1))]),
    lambda m: m.type(lambda t: t.nonintegertype(lambda n: n.keyword("real"))).declarators([lambda d: d.name(sv_identifier("c"))]),
    lambda m: m.type(lambda t: t.keywordtype(lambda k: k.keyword("string"))).declarators([lambda d: d.name(sv_identifier("d"))]),
    lambda m: m.type(lambda t: t.namedtype(lambda n: n.name(lambda x: x.identifier(sv_identifier("word_t"))))).declarators([lambda d: d.name(sv_identifier("e"))]),
    lambda m: m.type(lambda t: t.virtualinterfacetype(lambda v: v.interface(sv_identifier("bus")).parameters([SV.literal(8)]).modport(sv_identifier("mp")))).declarators(
        [lambda d: d.name(sv_identifier("f"))]),
    lambda m: m.type(lambda t: t.implicittype(lambda i: i.signing("signed"))).declarators([lambda d: d.name(sv_identifier("g"))]),
    lambda m: m.type(lambda t: t.enumtype(lambda e: e.base(sv_atom("int")).members([lambda n: n.name(sv_identifier("A")),
        lambda n: n.name(sv_identifier("B")).left(SV.literal(2)).value(SV.literal(1)), lambda n: n.name(sv_identifier("C")).left(SV.literal(0)).right(SV.literal(1))]))).declarators(
        [lambda d: d.name(sv_identifier("h"))]),
    lambda m: m.type(lambda t: t.typereference(lambda r: r.operand(sv_person))).declarators([lambda d: d.name(sv_identifier("i"))])]))
patterns_every = lambda u: u.structurepattern(lambda s: s.items([  # noqa: E731: '{a: .x, b: tagged Valid (.*), c: person.a}
    lambda i: i.patternmember(lambda m: m.name(sv_identifier("a")).pattern(lambda p: p.variablepattern(lambda v: v.name(sv_identifier("x"))))),
    lambda i: i.patternmember(lambda m: m.name(sv_identifier("b")).pattern(lambda p: p.taggedpattern(lambda t: t.member(sv_identifier("Valid")).pattern(
        lambda q: q.parenthesizedpattern(lambda r: r.pattern(lambda w: w.wildcardpattern(lambda x: x))))))),
    lambda i: i.patternmember(lambda m: m.name(sv_identifier("c")).pattern(sv_member))]))
assertions_every = lambda u: u.propertyspec(lambda s: s.clock(lambda k: k.events([  # noqa: E731
    lambda e: e.edge("posedge").expression(sv_person).condition(sv_person),
    lambda e: e.edge("negedge").expression(sv_person)])).disable(sv_person).property(lambda p: p.binaryproperty(lambda o: o.left(
    lambda l: l.binaryproperty(lambda q: q.left(
        lambda i: i.implicationproperty(lambda m: m.antecedent(lambda s1: s1.delaysequence(lambda ds: ds.first(
            lambda ps: ps.parenthesizedsequence(lambda pp: pp.sequence(lambda r: r.repetitionsequence(lambda rr: rr.sequence(sv_person).operator("*").count(
                lambda cr: cr.cyclerange(lambda c: c.low(SV.literal(1)).high(lambda dd: dd.dollarexpression(lambda z: z)))))).items([
                lambda it: it.assignmentexpression(lambda ae: ae.target(sv_person).operator("+=").value(SV.literal(1)))]))).steps([
            lambda st: st.delay(lambda cr: cr.cyclerange(lambda c: c.low(SV.literal(1)).high(SV.literal(3)))).sequence(
                lambda fm: fm.firstmatchsequence(lambda f: f.sequence(lambda bs: bs.binarysequence(lambda b2: b2.left(sv_person).operator("intersect").right(sv_person)))))]))).operator("|=>").consequent(
            lambda cs: cs.clockedsequence(lambda k: k.clock(lambda e: e).sequence(lambda ds: ds.delaysequence(lambda q2: q2.first(sv_person).steps([
                lambda st: st.delay(SV.literal(1)).sequence(lambda r: r.repetitionsequence(lambda rr: rr.sequence(sv_person).operator("->").count(SV.literal(2)))),
                lambda st: st.delay(sv_person).sequence(lambda r: r.repetitionsequence(lambda rr: rr.sequence(sv_person).operator("=").count(SV.literal(1))))]))))))).operator("and").right(
        lambda cp: cp.conditionalproperty(lambda c: c.condition(sv_person).consequence(
            lambda sp: sp.strengthproperty(lambda w: w.keyword("weak").sequence(lambda bs: bs.binarysequence(lambda b2: b2.left(sv_person).operator("or").right(sv_person))))).alternative(
            lambda ab: ab.abortproperty(lambda r: r.keyword("reject_on").condition(sv_person).operand(lambda cp2: cp2.caseproperty(lambda cc: cc.expression(sv_person).items([
                lambda it: it.expressions([SV.literal(1), SV.literal(2)]).body(lambda bp: bp.binaryproperty(lambda u2: u2.left(sv_person).operator("until").right(sv_person))),
                lambda it: it.body(lambda up: up.unaryproperty(lambda u3: u3.operator("not").operand(sv_person)))]))))))))).operator("or").right(
    lambda pp: pp.parenthesizedproperty(lambda q: q.property(lambda cl: cl.clockedproperty(lambda k: k.clock(lambda e: e.events([
        lambda ev: ev.edge("negedge").expression(sv_person)])).property(lambda bp: bp.binaryproperty(lambda o2: o2.left(
            lambda bs: bs.binarysequence(lambda t: t.left(sv_person).operator("throughout").right(sv_person))).operator("iff").right(
            lambda ip: ip.implicationproperty(lambda im: im.antecedent(lambda up: up.unaryproperty(lambda u4: u4.operator("nexttime").range(SV.literal(2)).operand(sv_person))).operator("#-#").consequent(sv_person))))))))))))
sv_holds = lambda e: lambda u: u.expressionconstraint(lambda c: c.expression(e))  # noqa: E731: a constraint of an expression
constraints_every = lambda u: u.constraintblock(lambda b: b.items([  # noqa: E731
    lambda i: i.expressionconstraint(lambda c: c.soft(True).expression(lambda d: d.distexpression(lambda e: e.value(sv_person).items([
        lambda t: t.value(SV.literal(0)).operator(":=").weight(SV.literal(40)),
        lambda t: t.value(SV.span(1, 3)).operator(":/").weight(SV.literal(60)),
        lambda t: t.operator(":/").weight(SV.literal(1))])))),  # soft person dist {0 := 40, [1:3] :/ 60, default :/ 1};
    lambda i: i.implicationconstraint(lambda c: c.condition(SV.binary(">", sv_person, 0)).body(sv_holds(sv_member))),
    lambda i: i.conditionalconstraint(lambda c: c.condition(sv_person).consequence(sv_holds(sv_member)).alternative(
        lambda e: e.constraintblock(lambda b: b.items([])))),
    lambda i: i.foreachconstraint(lambda f: f.value(sv_person).variables([lambda v: v.identifier(sv_identifier("k")),
                                                                         lambda v: v.emptyargument(lambda e: e)])
        .body(sv_holds(SV.binary("<", SV.identifier("k"), sv_member)))),
    lambda i: i.solvebeforeconstraint(lambda s: s.solve([sv_member]).before([sv_person])),
    lambda i: i.disablesoftconstraint(lambda d: d.target(sv_person)),
    lambda i: i.uniqueconstraint(lambda q: q.set([sv_person, SV.span(1, 2)])),
]))
preprocessor_every = lambda u: u.ifdefdirective(lambda d: d  # noqa: E731: `ifdef (A && !B) ... `elsif C ... `else ... `endif
    .condition(lambda c: c.parenthesizedexpression(lambda p: p.expression(SV.binary("&&", SV.identifier("A"), SV.unary("!", SV.identifier("B"))))))
    .items([
        lambda i: i.includedirective(lambda r: r.path("defs.svh")),
        lambda i: i.includedirective(lambda r: r.path("uvm.svh").system(True)),
        lambda i: i.definedirective(lambda r: r.name(sv_identifier("MAX")).function_like(True).parameters(
            [sv_identifier("a"), sv_identifier("b")]).body("((a) > (b) ? (a) : (b))")),
        lambda i: i.undefdirective(lambda r: r.name(sv_identifier("W"))),
        lambda i: i.timescaledirective(lambda r: r.unit("1ns").precision("1ps")),
        lambda i: i.defaultnettypedirective(lambda r: r.net_type("none")),
        lambda i: i.comment(lambda r: r.text(" low")),
        lambda i: i.comment(lambda r: r.text(" block ").block(True)),
        lambda i: i.macrousage(lambda r: r.name(sv_identifier("MAX")).arguments("1, 2")),
    ])
    .branches([lambda b: b.name(sv_identifier("C")).items([lambda i: i.otherdirective(lambda r: r.text("`resetall"))])])
    .alternative([lambda i: i.disabledtext(lambda r: r.text("skipped;"))]).has_else(True))
# TypeScript Types: every kind, the terms a type holds among them --- the text:
# any | unknown | number | bigint | boolean | string | symbol | object | never | void | undefined | null | intrinsic | this |
# A.B<T> | (T) | "s" | `a${T}b` | T[] | [a: T, T?, ...T[]] | (T & U) | (<T>(x: T) => x is T) | (abstract new () => T) |
# { a: T; m(x): T; (): T; new (): T; [k: T]: T; // c
# } | { [K in T as U]: K } | T[keyof T] | typeof this | import("m").A | (T extends (infer V)[] ? V : never)
ts_t = lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("T"))))  # noqa: E731: `T`
ts_annotated = lambda name: lambda p: p.identifier(lambda i: i.name(name).typeAnnotation(lambda a: a.typeAnnotation(ts_t)))  # noqa: E731
ts_types_every = lambda u: u.tsuniontype(lambda n: n.types([  # noqa: E731: every kind, written through the builders
    *[lambda k, keyword=keyword: getattr(k, keyword)(lambda h: h) for keyword in TST.KEYWORDS],
    lambda k: k.tstypereference(lambda r: r.typeName(lambda m: m.tsqualifiedname(lambda q: q.left(lambda l: l.identifier(
        lambda i: i.name("A"))).right(lambda i: i.name("B")))).typeArguments(lambda a: a.params([ts_t]))),
    lambda k: k.tsparenthesizedtype(lambda p: p.typeAnnotation(ts_t)),
    lambda k: k.tsliteraltype(lambda p: p.literal(lambda li: li.literal(lambda v: v.value(lambda x: x.as_scalar("s"))))),
    lambda k: k.tstemplateliteraltype(lambda p: p.quasis([lambda q: q.raw("a"), lambda q: q.raw("b").tail(True)]).types([ts_t])),
    lambda k: k.tsarraytype(lambda a: a.elementType(ts_t)),
    lambda k: k.tstupletype(lambda p: p.elementTypes([
        lambda e: e.tsnamedtuplemember(lambda m: m.label(lambda i: i.name("a")).elementType(ts_t)),
        lambda e: e.tsoptionaltype(lambda o: o.typeAnnotation(ts_t)),
        lambda e: e.tsresttype(lambda o: o.typeAnnotation(lambda a: a.tsarraytype(lambda b: b.elementType(ts_t))))])),
    lambda k: k.tsintersectiontype(lambda p: p.types([ts_t, lambda t: t.tstypereference(lambda r: r.typeName(
        lambda m: m.identifier(lambda i: i.name("U"))))])),
    lambda k: k.tsfunctiontype(lambda f: f.typeParameters(lambda d: d.params([lambda p: p.name(lambda i: i.name("T"))])).params([
        ts_annotated("x")]).returnType(lambda a: a.typeAnnotation(lambda p: p.tstypepredicate(lambda q: q.parameterName(
            lambda m: m.identifier(lambda i: i.name("x"))).typeAnnotation(lambda b: b.typeAnnotation(ts_t)))))),
    lambda k: k.tsconstructortype(lambda f: f.abstract(True).returnType(lambda a: a.typeAnnotation(ts_t))),
    lambda k: k.tstypeliteral(lambda q: q.members([
        lambda m: m.tspropertysignature(lambda s: s.key(lambda y: y.identifier(lambda i: i.name("a"))).typeAnnotation(lambda a: a.typeAnnotation(ts_t))),
        lambda m: m.tsmethodsignature(lambda s: s.key(lambda y: y.identifier(lambda i: i.name("m"))).params([
            lambda p: p.identifier(lambda i: i.name("x"))]).returnType(lambda a: a.typeAnnotation(ts_t))),
        lambda m: m.tscallsignaturedeclaration(lambda s: s.returnType(lambda a: a.typeAnnotation(ts_t))),
        lambda m: m.tsconstructsignaturedeclaration(lambda s: s.returnType(lambda a: a.typeAnnotation(ts_t))),
        lambda m: m.tsindexsignature(lambda s: s.parameters([ts_annotated("k")]).typeAnnotation(lambda a: a.typeAnnotation(ts_t))),
        lambda m: m.comment(lambda c: c.text(" c"))])),
    lambda k: k.tsmappedtype(lambda m: m.key(lambda i: i.name("K")).value(ts_t).nameType(lambda t: t.tstypereference(
        lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("U"))))).body(lambda t: t.tstypereference(
            lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("K")))))),
    lambda k: k.tsindexedaccesstype(lambda a: a.objectType(ts_t).indexType(lambda i: i.tstypeoperator(
        lambda o: o.operator("keyof").typeAnnotation(ts_t)))),
    lambda k: k.tstypequery(lambda q: q.exprName(lambda e: e.thisexpression(lambda h: h))),
    lambda k: k.tsimporttype(lambda i: i.source(lambda s: s.literal(lambda li: li.value(lambda x: x.as_scalar("m")))).qualifier(
        lambda q: q.identifier(lambda j: j.name("A")))),
    lambda k: k.tsconditionaltype(lambda c: c.checkType(ts_t).extendsType(lambda e: e.tsarraytype(lambda a: a.elementType(
        lambda i: i.tsinfertype(lambda f: f.typeParameter(lambda p: p.name(lambda n: n.name("V"))))))).trueType(
        lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("V"))))).falseType(
        lambda t: t.tsneverkeyword(lambda h: h))),
]))
# TypeScript Patterns: every kind, the terms a target may be among them --- the text:
# [{ a: person = 1, ...person }, , person.x, private person]
ts_patterns_every = lambda u: u.arraypattern(lambda a: a.elements([  # noqa: E731: every kind, written through the builders
    lambda e: e.objectpattern(lambda o: o.properties([
        lambda p: p.property(lambda q: q.propertyKind("init").key(lambda k: k.identifier(lambda i: i.name("a"))).value(
            lambda v: v.assignmentpattern(lambda s: s.left(ts_person).right(lambda r: r.literal(lambda li: li.value(
                lambda x: x.as_scalar(1.0))))))),
        lambda p: p.restelement(lambda r: r.argument(ts_person))])),
    lambda e: e.elision(lambda h: h),
    lambda e: e.memberexpression(lambda m: m.object(ts_person).property(lambda y: y.identifier(lambda i: i.name("x")))),
    lambda e: e.tsparameterproperty(lambda t: t.accessibility("private").parameter(ts_person))]))
# The regular expression dialects, each of every kind, their properties named as regexpp names ECMAScript's.
# Python RegExp --- the text: (?:a|)(?P<n>[a-z\d]+?)(?<!b)(?P=n)(?(1)a)(?>.)b*+^(?#c)|(?i-m:a)
pyr_every = lambda u: u.pattern(lambda p: p.alternatives([  # noqa: E731
    lambda s: s.alternative(lambda a: a.elements([
        lambda e: e.group(lambda g: g.alternatives([
            lambda x: x.alternative(lambda y: y.elements([lambda c: c.character(lambda v: v.value(97))])),
            lambda x: x.alternative(lambda y: y)])),
        lambda e: e.capturinggroup(lambda g: g.name("n").alternatives([lambda x: x.alternative(lambda y: y.elements([
            lambda q: q.quantifier(lambda r: r.min(1).greedy(False).element(lambda k: k.characterclass(lambda c: c.elements([
                lambda r: r.characterclassrange(lambda n: n.min(lambda c: c.character(lambda v: v.value(97))).max(
                    lambda c: c.character(lambda v: v.value(122)))),
                lambda s: s.characterset(lambda v: v.setKind("digit"))]))))]))])),
        lambda e: e.assertion(lambda t: t.assertionKind("lookbehind").negate(True).alternatives([
            lambda x: x.alternative(lambda y: y.elements([lambda c: c.character(lambda v: v.value(98))]))])),
        lambda e: e.backreference(lambda r: r.ref("n")),
        lambda e: e.conditional(lambda c: c.ref(1).yes(lambda x: x.alternative(lambda y: y.elements([
            lambda c: c.character(lambda v: v.value(97))])))),
        lambda e: e.atomicgroup(lambda g: g.alternatives([lambda x: x.alternative(lambda y: y.elements([
            lambda s: s.characterset(lambda v: v.setKind("any"))]))])),
        lambda e: e.quantifier(lambda q: q.min(0).possessive(True).element(lambda c: c.character(lambda v: v.value(98)))),
        lambda e: e.assertion(lambda t: t.assertionKind("start")),
        lambda e: e.comment(lambda c: c.text("c"))])),
    lambda s: s.alternative(lambda a: a.elements([
        lambda e: e.flags(lambda f: f.add("i").remove("m").alternatives([lambda x: x.alternative(lambda y: y.elements([
            lambda c: c.character(lambda v: v.value(97))]))]))]))]))
# TypeScript RegExp --- the text: /(?i:a)(?<n>b{1,}?)[a-c\d][\q{x|y}&&[x]][\w--a]\p{L}\b(?=c)\k<n>|$/v
ts_char = lambda c: lambda e: e.character(lambda h: h.value(ord(c)))  # noqa: E731
ts_seq = lambda *items: lambda e: e.alternative(lambda a: a.elements(list(items)))  # noqa: E731
tsr_every = lambda u: u.regexpliteral(lambda l: l.pattern(lambda p: p.pattern(lambda q: q.alternatives([  # noqa: E731
    ts_seq(lambda e: e.group(lambda g: g.modifiers(lambda m: m.modifiers(lambda s: s.add(lambda f: f.ignoreCase(True)))).alternatives(
               [ts_seq(ts_char("a"))])),
           lambda e: e.capturinggroup(lambda g: g.name("n").alternatives([ts_seq(lambda e: e.quantifier(lambda q: q.raw("b{1,}?").min(1).greedy(
               False).element(ts_char("b"))))])),
           lambda e: e.characterclass(lambda c: c.elements([lambda e: e.characterclassrange(lambda r: r.min(ts_char("a")).max(ts_char("c"))),
                                                            lambda e: e.characterset(lambda s: s.setKind("digit"))])),
           lambda e: e.expressioncharacterclass(lambda x: x.expression(lambda e: e.classintersection(lambda i: i.left(
               lambda e: e.classstringdisjunction(lambda d: d.alternatives([lambda e: e.stringalternative(lambda s: s.elements([ts_char("x")])),
                                                                            lambda e: e.stringalternative(lambda s: s.elements([ts_char("y")]))]))).right(
               lambda e: e.characterclass(lambda c: c.elements([ts_char("x")])))))),
           lambda e: e.expressioncharacterclass(lambda x: x.expression(lambda e: e.classsubtraction(lambda s: s.left(
               lambda e: e.characterset(lambda t: t.setKind("word"))).right(ts_char("a"))))),
           lambda e: e.characterset(lambda s: s.setKind("property").key("L")), lambda e: e.assertion(lambda a: a.assertionKind("word")),
           lambda e: e.assertion(lambda a: a.assertionKind("lookahead").alternatives([ts_seq(ts_char("c"))])),
           lambda e: e.backreference(lambda b: b.ref("n"))),
    ts_seq(lambda e: e.assertion(lambda a: a.assertionKind("end")))]))).flags(lambda f: f.unicodeSets(True)))
# Cpp RegExp --- ECMAScript's: ^(a|.)[^a-[.ch.][:alpha:][=e=]\d]{1,3}.$|(?:b)(?!c)x*?\1\b
regexp_char = lambda c: lambda u: u.character(lambda r: r.value(ord(c)))  # noqa: E731: a character, of every RegExp dialect here
regexp_alt = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))  # noqa: E731
xr_every = lambda u: u.regex(lambda r: r.flags(["ECMAScript"]).pattern(XR.spec(lambda p: p.pattern(lambda w: w.alternatives([  # noqa: E731
    regexp_alt(
        lambda e: e.assertion(lambda a: a.assertionKind("start")),
        lambda e: e.capturinggroup(lambda g: g.alternatives([regexp_alt(regexp_char("a")), regexp_alt(regexp_char("."))])),
        lambda e: e.quantifier(lambda m: m.min(1).max(3).element(lambda c: c.characterclass(lambda k: k.negate(True).elements([
            lambda x: x.characterclassrange(lambda y: y.min(regexp_char("a")).max(lambda z: z.collatingelement(lambda n: n.name("ch")))),
            lambda x: x.exclass(lambda n: n.name("alpha")), lambda x: x.equivalence(lambda n: n.name("e")),
            lambda x: x.characterset(lambda n: n.setKind("digit"))])))),
        lambda e: e.characterset(lambda n: n.setKind("any")), lambda e: e.assertion(lambda a: a.assertionKind("end"))),
    regexp_alt(
        lambda e: e.group(lambda g: g.alternatives([regexp_alt(regexp_char("b"))])),
        lambda e: e.assertion(lambda a: a.assertionKind("lookahead").negate(True).alternatives([regexp_alt(regexp_char("c"))])),
        lambda e: e.quantifier(lambda m: m.raw("*?").min(0).greedy(False).element(regexp_char("x"))),
        lambda e: e.backreference(lambda b: b.ref(1)), lambda e: e.assertion(lambda a: a.assertionKind("word")))])))))
# POSIX BRE --- grep's list of two patterns: ^\(a\)[^a-[.ch.][:alpha:][=e=]]\{1,3\}.\1$, then b*
posix_class = lambda c: c.characterclass(lambda k: k.negate(True).elements([  # noqa: E731: [^a-[.ch.][:alpha:][=e=]]
    lambda x: x.characterclassrange(lambda y: y.min(regexp_char("a")).max(lambda z: z.collatingelement(lambda n: n.name("ch")))),
    lambda x: x.exclass(lambda n: n.name("alpha")), lambda x: x.equivalence(lambda n: n.name("e"))]))
bre_every = lambda u: u.patternlist(lambda q: q.patterns([  # noqa: E731
    lambda v: v.pattern(lambda w: w.alternatives([regexp_alt(
        lambda e: e.assertion(lambda a: a.assertionKind("start")),
        lambda e: e.capturinggroup(lambda g: g.alternatives([regexp_alt(regexp_char("a"))])),
        lambda e: e.quantifier(lambda m: m.min(1).max(3).element(posix_class)),
        lambda e: e.characterset(lambda n: n.setKind("any")), lambda e: e.backreference(lambda b: b.ref(1)),
        lambda e: e.assertion(lambda a: a.assertionKind("end")))])),
    lambda v: v.pattern(lambda w: w.alternatives([regexp_alt(lambda e: e.quantifier(lambda m: m.min(0).element(regexp_char("b"))))]))]))
# POSIX ERE --- grep -E's list of two patterns: ^(a|.)[^a-[.ch.][:alpha:][=e=]]{1,3}?$, then b+{0,1}
ere_every = lambda u: u.patternlist(lambda q: q.patterns([  # noqa: E731
    lambda v: v.pattern(lambda w: w.alternatives([regexp_alt(
        lambda e: e.assertion(lambda a: a.assertionKind("start")),
        lambda e: e.capturinggroup(lambda g: g.alternatives([regexp_alt(regexp_char("a")),
                                                            regexp_alt(lambda s: s.characterset(lambda n: n.setKind("any")))])),
        lambda e: e.quantifier(lambda m: m.min(1).max(3).greedy(False).element(posix_class)),
        lambda e: e.assertion(lambda a: a.assertionKind("end")))])),
    lambda v: v.pattern(lambda w: w.alternatives([regexp_alt(lambda e: e.quantifier(lambda m: m.min(0).max(1).element(
        lambda f: f.quantifier(lambda n: n.min(1).element(regexp_char("b"))))))]))]))
samples = {  # one term per dialect, with every kind
    "Patterns": term(E.operation("and", basic, E.list_(1, E.mapping(("k", person.items)).item("k")).all(  # a quantifier too
        "x", E.variable("x").gt(0)))),
    "Python3": term(P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("person"), P.ifexp(  # every kind
        P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1, ("<", 2))),
                                P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2, 3), P.namedexpr("q", 0)),
        P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), P.tuple_(P.slice_(1, None, -1), P.constant(...))),
               P.starred(P.list_(1)), P.keyword("k", P.dict_(("k", 2), (None, P.name("a"))))),
        P.call("f", P.generatorexp(P.subscript(P.name("p"), 0), P.comprehension("p", P.attribute(P.name("a"), "z"), (P.name("p"),))),
               P.listcomp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("r"))), P.name("a"), is_async=True)),
               P.setcomp(P.name("p"), P.comprehension("p", P.name("a"))), P.dictcomp(P.name("p"), None, P.comprehension("p", P.name("a"))),
               P.set_(P.constant(None), P.constant(1j)), P.lambda_(P.name("b"), (P.arg("b", 1),)), P.yield_(), P.yieldfrom(P.name("a")),
               P.await_(P.name("a")), P.concatenatedstring("x", P.joinedstr(("y", P.formattedvalue(P.name("a"), "r")))),
               P.templatestr((P.interpolation(P.name("a")),)), P.parenthesized(P.name("a"))))),
        alias="f"), alias="np"), P.DIALECT),
    "MATLAB": term(M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("person"), "age")),
                                               M.call("isfield", "x", M.index(M.array(M.arrayfun("p", M.identifier("person"), M.identifier("p")), 2), 1)))), M.DIALECT),
    "Excel": term(X.let_("a", X.field(X.name("person"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
                                                                              X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2)))))), X.DIALECT),
    "LaTeX": term(L.where("a", L.member(L.symbol("person"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18), L.binary(
        "\\lor", L.unary("\\lnot", L.function("has", L.symbol("person"), "email")), L.binary(">", L.frac(L.symbol("a"), 2), 1.5)))), L.DIALECT),
    "C": term(c_every, C.DIALECT),
    "Cpp": term(cpp_every, CPP.DIALECT),
    "C Types": term(enumeration(CT, C), CT),
    "Cpp Types": term(enumeration(XT, CPP), XT),
    "C Preprocessor": term(c_preprocessor_every, CPRE),
    "SystemVerilog": term(sv_every, SV.DIALECT),
    "SystemVerilog Types": term(types_every, SVT.DIALECT),
    "SystemVerilog Patterns": term(patterns_every, SVP.DIALECT),
    "SystemVerilog Assertions": term(assertions_every, SVA.DIALECT),
    "SystemVerilog Constraints": term(constraints_every, SVC.DIALECT),
    "SystemVerilog Preprocessor": term(preprocessor_every, SVPP.DIALECT),
    "TypeScript5": term(typescript5, TS.DIALECT),
    "TypeScript Types": term(ts_types_every, TST.DIALECT),
    "TypeScript Patterns": term(ts_patterns_every, TSP.DIALECT),
    "Python Types": term(lambda u: u.typevar(lambda t: t.name("T").bound(  # T: **P = [int, person] = *Ts = *person
        lambda b: b.paramspec(lambda s: s.name("P").default_value(P.list_(P.name("int"), P.name("person"))))).default_value(
        lambda d: d.typevartuple(lambda s: s.name("Ts").default_value(P.starred(P.name("person")))))), PYT.DIALECT),
    "Python Patterns": term(lambda u: u.matchor(lambda o: o.patterns([  # person.Point(0, y=[a, *rest]) | {'k': None, **kw} | (person.RED as c)
        lambda p: p.matchclass(lambda c: c.cls(P.attribute(P.name("person"), "Point")).patterns([
            lambda v: v.matchvalue(lambda m: m.value(P.constant(0)))]).kwd_attrs(["y"]).kwd_patterns([
            lambda s: s.matchsequence(lambda q: q.delimiters("[]").patterns([lambda a: a.matchas(lambda m: m.name("a")),
                                                                          lambda r: r.matchstar(lambda m: m.name("rest"))]))])),
        lambda p: p.matchmapping(lambda m: m.keys([P.constant("k")]).patterns([lambda s: s.matchsingleton(lambda v: v.value("None"))]).rest("kw")),
        lambda p: p.matchas(lambda m: m.pattern(lambda v: v.matchvalue(lambda w: w.value(P.attribute(P.name("person"), "RED")))).name("c"))])),
        PYP.DIALECT),
    "Python RegExp": term(pyr_every, PYR.DIALECT),
    "TypeScript RegExp": term(tsr_every, TSR.DIALECT),
    "Cpp RegExp": term(xr_every, XR),
    "POSIX BRE": term(bre_every, PXB.DIALECT),
    "POSIX ERE": term(ere_every, PXE.DIALECT),
}
DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, CPP.DIALECT, SV.DIALECT, SVT.DIALECT,
            SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT, TS.DIALECT, TST.DIALECT, TSP.DIALECT, PYT.DIALECT, PYP.DIALECT,
            PYR.DIALECT, TSR.DIALECT]
FAMILY = DIALECTS[7:13]  # SystemVerilog's six dialects, each reading the others' terms
TS_FAMILY = [*DIALECTS[13:16], TSR.DIALECT]  # TypeScript's four, each reading the others' terms
PYTHON = (P.DIALECT, PYT.DIALECT, PYP.DIALECT, PYR.DIALECT)  # Python's four, each reading the others' terms
READER = {d.union: d for d in DIALECTS + C_FAMILY + POSIX}  # the dialect whose union holds a term
SIMPLIFIERS = {E.DIALECT: V.SIMPLIFIER, P.DIALECT: PV.SIMPLIFIER, M.DIALECT: MV.SIMPLIFIER, X.DIALECT: XV.SIMPLIFIER,
               C.DIALECT: CV.SIMPLIFIER, CPP.DIALECT: CPV.SIMPLIFIER, SV.DIALECT: SVV.SIMPLIFIER,
               TS.DIALECT: TV.SIMPLIFIER}  # LaTeX has none
INVALID = {"error", "missing"}  # the framework's kinds of invalid syntax, which every dialect's `Term` holds

<a id="FRM-01"></a>
## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
assert all(isinstance(simplify, FV.Simplifier) for simplify in SIMPLIFIERS.values())
for dialect in DIALECTS + C_FAMILY + POSIX:
    assert isinstance(dialect, F.Dialect) and all(isinstance(kind, F.Kind) for kind in dialect.kinds().values())
    for node in F.walk(samples[dialect.name()]):  # a term of its own union, nested in its kinds or in shared ones
        if dialect.reads():  # or a term of a dialect of its family, which it reads
            reader = READER[F.dialect_of(node)]
            assert F.is_term(node) and reader in (dialect, *dialect.reads()) and dialect.kind(node)[0] == F.kind_of(node)[0]
            continue
        assert F.is_term(node) and F.dialect_of(node) == dialect.union and F.kind_of(node)[0] in dialect.kinds()
assert [d.name() for d in DIALECTS] == ["Patterns", "Python3", "MATLAB", "Excel", "LaTeX", "C", "Cpp", "SystemVerilog",
                                     "SystemVerilog Types", "SystemVerilog Patterns", "SystemVerilog Assertions",
                                     "SystemVerilog Constraints", "SystemVerilog Preprocessor", "TypeScript5",
                                     "TypeScript Types", "TypeScript Patterns", "Python Types", "Python Patterns",
                                     "Python RegExp", "TypeScript RegExp"]
# A sublanguage's dialect is named by its family and its own name; its tag, its name lowercased, a space an underscore.
assert [d.tag for d in FAMILY] == ["systemverilog", "systemverilog_types", "systemverilog_patterns",
                                   "systemverilog_assertions", "systemverilog_constraints", "systemverilog_preprocessor"]
assert [F.dialect_name(path) for path in ("SystemVerilog.Types", "C.RegExp", "Python.Python3", "C.Expressions", "POSIX.BRE")] == [
    "SystemVerilog Types", "C RegExp", "Python3", "C", "POSIX BRE"] and F.dialect_tag("SystemVerilog Types") == "systemverilog_types"
assert F.SUBLANGUAGES == ("Types", "Patterns", "Assertions", "Constraints", "Preprocessor", "RegExp", "BRE", "ERE")
for family in (FAMILY, TS_FAMILY, PYTHON):  # each reads the others' terms, in its family's order
    for dialect in family:
        assert dialect.reads() == tuple(d for d in family if d is not dialect) and dialect.reads() is dialect.reads()
assert [d.tag for d in TS_FAMILY] == ["typescript5", "typescript_types", "typescript_patterns", "typescript_regexp"]
# The retrofit's new dialects: each a package exporting its install and define, its singleton, its `Term`, its holder,
# its printer and its `reads`, each family's dialects reading one another.
NEW = [package.Expressions.DIALECT for package in RETROFIT]
PT, PP, TT, TP = NEW[:4]
PR, TR = NEW[7:9]  # the regular expression dialects, installed last: Python's, TypeScript's, C++'s, XR, and POSIX's
assert NEW[4:7] + NEW[9:10] == C_FAMILY and NEW[10:] == POSIX  # the C family's: C Types, Cpp Types, C Preprocessor, Cpp RegExp
assert [d.name() for d in NEW] == ["Python Types", "Python Patterns", "TypeScript Types", "TypeScript Patterns", "C Types",
                                   "Cpp Types", "C Preprocessor", "Python RegExp", "TypeScript RegExp", "Cpp RegExp",
                                   "POSIX BRE", "POSIX ERE"]
assert [d.tag for d in NEW] == ["python_types", "python_patterns", "typescript_types", "typescript_patterns", "c_types",
                                "cpp_types", "c_preprocessor", "python_regexp", "typescript_regexp", "cpp_regexp",
                                "posix_bre", "posix_ere"]
READS = {P.DIALECT: (PT, PP, PR), PT: (P.DIALECT, PP, PR), PP: (P.DIALECT, PT, PR), PR: (P.DIALECT, PT, PP),
         TS.DIALECT: (TT, TP, TR), TT: (TS.DIALECT, TP, TR), TP: (TS.DIALECT, TT, TR), TR: (TS.DIALECT, TT, TP),
         C.DIALECT: (CT, CPRE), CT: (C.DIALECT, CPRE), CPP.DIALECT: (XT, CPRE, XR), XT: (CPP.DIALECT, CPRE, XR),
         CPRE: (C.DIALECT, CT, CPP.DIALECT, XT), XR: (CPP.DIALECT, XT, CPRE, *POSIX), POSIX[0]: (POSIX[1],),
         POSIX[1]: (POSIX[0],)}  # C++'s read C's preprocessor, and none of C's other dialects; the preprocessor reads C's
# and C++'s expressions and types, a condition's and an enumeration's; C++'s dialects read one another, its regular
# expressions among them, as each family's do, and those read POSIX's too, which a regex holds; POSIX's read each other's
assert all(d.reads() == READS.get(d, ()) for d in DIALECTS + NEW if d not in FAMILY)
for package, dialect in zip(RETROFIT, NEW):  # each its package, with its printer, its own kinds, its holder, which
    # names its ABI
    assert package.__all__ == ["Expressions", "Text", "define", "install"] and package.install is package.Expressions.install
    assert package.define is package.Expressions.define and list(dialect.kinds()) == [
        *package.Expressions.TAGS, *(["identifier"] if dialect is TT else []), "error", "missing"]  # TypeScript Types' own identifier
    holder = store.registered(f"Expressions.{dialect.path}.Expression")
    assert holder.properties["root"].schema is store.registered(dialect.union) and list(holder.adjacencies) == ["abi"]
    assert holder.refines is store.registered("Expressions.Framework.Expression") and package.Text.ToText(samples[dialect.name()])
# A kind's owner, whose name its refusals and its printing of invalid syntax give: the dialect whose `Term` holds it,
# or, for a kind that `Term` lists and a dialect it reads declares, that one; a term of a dialect it does not read, itself.
typevar = term(lambda u: u.typevar(lambda t: t.name("T")), PT)
listed = term(TT.make(F.Form("literal", {"value": 1.0})), TT)  # TypeScript5's literal, which Types' `Term` lists
assert P.DIALECT.owner(typevar) is PT and PT.owner(typevar) is PT and C.DIALECT.owner(typevar) is C.DIALECT
assert TT.owner(listed) is TS.DIALECT and TS.DIALECT.owner(listed) is TS.DIALECT
for package in (SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor):  # each its package, its exports alike
    assert package.__all__ == ["Expressions", "Text", "define", "install"] and package.install is package.Expressions.install
    assert package.define is package.Expressions.define and package.Expressions.DIALECT in FAMILY
    holder = store.registered(f"Expressions.{package.Expressions.DIALECT.path}.Expression")  # each its holder
    assert holder.refines is store.registered("Expressions.Framework.Expression") and list(holder.adjacencies) == ["abi"]
assert SystemVerilog.__all__ == ["Domains", "Expressions", "Simplifiers", "Text", "define", "install"]
ORDER = DIALECTS[:14] + NEW  # in install order: each new dialect after its family's
assert repr(E.DIALECT) == "<dialect Patterns>" and [d.tag for d in ORDER] == [b.name for b in Term.branches]
assert [d.union for d in ORDER] == [b.schema.name for b in Term.branches]  # the union of the dialects' `Term`s
# Every dialect a branch of the framework's kind union of dialects: the standard dialects first, which have no terms;
# the framework's union of kinds is of the dialects' unions, each holding its dialect's kinds.
Dialects, DomainKind = store.registered("Expressions.Framework.Dialect"), store.registered("Expressions.Framework.DomainKind")
assert [b.name for b in Dialects.branches] == ["isoiec11404", "ieee754", "ieee1164", "unicode", *[d.tag for d in DIALECTS[:7]],
                                             "amd64", "aapcs64", *[d.tag for d in ORDER[7:]]]  # in install order: the ABIs, terms none
assert [b.schema.singleton for b in Dialects.branches[20:]] == [f"Expressions.{d.path}" for d in NEW]
assert [b.schema.singleton for b in Dialects.branches[13:19]] == [f"Expressions.{d.path}" for d in FAMILY]
assert [b.schema.singleton for b in Dialects.branches[11:13]] == ["Expressions.SystemV.AMD64", "Expressions.Arm.AAPCS64"]
assert [F.dialect_name(b.schema.singleton.removeprefix("Expressions.")) for b in Dialects.branches[:4]] == [
    "ISOIEC11404", "IEEE754", "IEEE1164", "Unicode"]
assert {b.name: [k.name for k in b.schema.branches] for b in DomainKind.branches} == {
    "isoiec11404": ["boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"], "ieee754": ["ieee754"],
    "ieee1164": ["ieee1164"], "unicode": ["unicode"], "python3": ["str", "nonetype", "ellipsis"], "matlab": ["string"],
    "excel": ["text", "error"],
    "c": ["nullptr_t"], "cpp": ["nullptr_t"], "systemverilog": ["logic", "bit"], "typescript5": ["string", "null"]}
assert [b.schema.name for b in DomainKind.branches[4:]] == [  # each dialect's own union of kinds, named by its path
    d.union.removesuffix(".Term") + ".DomainKind" for d in DIALECTS if d.name() not in ("Patterns", "LaTeX")
    and d not in FAMILY[1:] + TS_FAMILY[1:] + list(PYTHON[1:])]
assert all(list(b.schema.adjacencies) == ["domains", "operations", "standards"] for b in Dialects.branches)
assert list(X.DIALECT.kinds()) == ["literal", "variable", "cell", "let", "call", "infix", "prefix", "field", "map", "error",
                                   "missing"]
HELD = {C.DIALECT: {"typeid", "definedexpression"}, CPP.DIALECT: {"typeid", "definedexpression"}}  # its Types' and
# the preprocessor's kinds its subterms may be, which C's and C++'s `Term`s list
for dialect in DIALECTS:  # each union's branches are its dialect's kinds, named plainly; shared kinds, the framework's
    union = store.registered(dialect.union)
    if dialect is PT:  # Python Types', Python3's too: a type expression is any expression
        assert {b.name for b in union.branches} == set(PT.kinds()) | set(P.DIALECT.kinds())
        continue
    if dialect in FAMILY:  # and in SystemVerilog's family, the kinds of the dialects it reads that its subterms may be
        own = set(dialect.kinds()) | ({"macrousage"} if dialect is not SVPP.DIALECT else set())
        own |= {SVA.DIALECT: {"distexpression"}, SVC.DIALECT: {*SVPP.DIRECTIVES, "comment"}}.get(dialect, set())  # held
        assert {b.name for b in union.branches} == own | (set(SV.DIALECT.kinds()) if dialect is not SVPP.DIALECT else set()), dialect.name()
        continue
    if dialect in TS_FAMILY:  # and in TypeScript's, the terms of the dialects it reads that its subterms may be
        held = {TS.DIALECT: set(TSP.TAGS[:3]), TST.DIALECT: set(TST.TERMS), TSP.DIALECT: set(TSP.TERMS), TSR.DIALECT: set()}[dialect]
        assert {b.name for b in union.branches} == set(dialect.kinds()) - ({"identifier"} if dialect is TST.DIALECT else set()) | held
        continue
    assert {b.name for b in union.branches} == set(dialect.kinds()) | HELD.get(dialect, set()), dialect.name()
    shared = {b.schema.name.rsplit(".", 1)[1].lower() for b in union.branches if b.schema.name.rpartition(".")[0] == "Expressions.Framework"}
    assert shared <= {"literal", "call", "variable", "let", "list", "mapping", "error", "missing"} and INVALID <= shared
C_UNION, CPP_UNION = store.registered(C.DIALECT.union), store.registered(CPP.DIALECT.union)
assert [b.name for b in C_UNION.branches] == [*list(C.DIALECT.kinds())[:-2], "typeid", "error", "missing", "definedexpression"]
assert [b.name for b in CPP_UNION.branches] == [*list(CPP.DIALECT.kinds())[:-2], "typeid", "error", "missing", "definedexpression"]
assert {b.name for b in C_UNION.branches} - {b.name for b in CPP_UNION.branches} == {"variable", "genericselection", "compoundliteralexpression"}
assert "variable" not in CPP.DIALECT.kinds()  # C++ writes a name only as an idexpression, its grammar's one kind
assert {b.name for b in CPP_UNION.branches} - {b.name for b in C_UNION.branches} == {  # C++ is not a superset of C
    "userdefinedliteral", "idexpression", "thisexpression", "lambdaexpression", "foldexpression", "requiresexpression",
    "functionalcastexpression", "namedcastexpression", "typeidexpression", "awaitexpression", "sizeofpackexpression",
    "noexceptexpression", "newexpression", "deleteexpression", "throwexpression", "yieldexpression", "packexpansion"}
assert all(b.schema.name.startswith("Expressions.C.Expressions.") for b in C_UNION.branches if b.name not in (
    "variable", "error", "missing", "typeid", "definedexpression"))
assert all(b.schema.name.startswith("Expressions.Cpp.Expressions.") for b in CPP_UNION.branches if b.name not in (
    "error", "missing", "typeid", "definedexpression"))
assert {b.name: b.schema.name for b in C_UNION.branches if b.name in HELD[C.DIALECT]} == {  # each its own dialect's
    "typeid": "Expressions.C.Types.TypeId", "definedexpression": "Expressions.C.Preprocessor.DefinedExpression"}
assert list(E.DIALECT.kinds()) == ["literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]
assert [kind.role for kind in E.DIALECT.kinds().values()] == [
    F.LITERAL, F.APPLICATION, F.REFERENCE, F.BINDING, F.APPLICATION, F.APPLICATION, F.QUANTIFIER, F.INVALID, F.INVALID]
for domain in [D.Anything, D.OfTypes("int", int), D.OfValues("odd", lambda v: v % 2 == 1), D.OfUnion(D.Anything)]:
    assert isinstance(domain, D.Domain)
for signature in [D.Function((), D.Anything), D.Same(1, (D.Anything,)), D.Overloaded(D.Function((), D.Anything))]:
    assert isinstance(signature, D.Signature)
for translator in Translators.TRANSLATORS:
    assert isinstance(translator, FT.Translator) and isinstance(translator.inverse(), FT.Pairwise)
# The package exports the framework alone; each dialect, Patterns among them, is imported from its own path.
import mbse.Expressions as package
assert package.__all__ == ["Framework"] and package.Framework is Framework
assert not any(hasattr(package, name) for name in ("Domains", "Simplifiers", "Expressions", "Partials"))

<a id="FRM-02"></a>
## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
for dialect in DIALECTS + C_FAMILY + POSIX:
    kinds = set()
    for node in F.walk(samples[dialect.name()]):
        form = F.form(node)
        kinds.add(form.kind)
        made = term(READER[F.dialect_of(node)].make(form))  # made by the dialect whose union holds it
        assert F.same(made, node) and F.form(made).kind == form.kind and made is not node
    if dialect.reads():  # SystemVerilog's sample holds its Types' too, its sublanguages' those of Expressions, and the C
        # family's those of the dialects each reads
        assert set(dialect.kinds()) - INVALID <= kinds, (dialect.name(), kinds)
        continue
    assert kinds == set(dialect.kinds()) - INVALID, (dialect.name(), kinds)  # the samples have every valid kind
# SystemVerilog's sample holds every kind of its Expressions and Types, and the Preprocessor's macro usage; a dialect of
# its family makes a kind of a dialect it reads, where its `Term` lists it, and refuses one it does not.
assert {F.kind_of(n)[0] for n in F.walk(samples["SystemVerilog"])} == set(SV.TAGS) | {"integeratomtype", "macrousage"}
assert {F.kind_of(n)[0] for n in F.walk(samples["SystemVerilog Types"])} >= set(SVT.TAGS)
assert F.form(term(SVT.DIALECT.make(F.Form("literal", {"value": 8})))) == F.Form("literal", {"value": 8})  # Types' Term lists it
assert SVT.DIALECT.kind_named("literal") is SV.DIALECT.kinds()["literal"] and SV.DIALECT.kind_named("structtype") is SVT.DIALECT.kinds()["structtype"]
with raises(ValueError, match="SystemVerilog Types has no kind 'call'"):
    SVT.DIALECT.make(F.Form("call"))
assert {F.kind_of(n)[0] for n in F.walk(samples["TypeScript Types"])} >= set(TST.TAGS)  # and TypeScript's alike
assert {F.kind_of(n)[0] for n in F.walk(samples["TypeScript Patterns"])} >= set(TSP.TAGS)
assert F.form(term(TST.DIALECT.make(F.Form("literal", {"value": 1.0})), TST.DIALECT)) == F.Form("literal", {"value": 1.0})
assert TST.DIALECT.kind_named("literal") is TS.DIALECT.kinds()["literal"] and TS.DIALECT.kind_named("tsuniontype") is TST.DIALECT.kinds()["tsuniontype"]
assert TS.DIALECT.kind_named("arraypattern") is TSP.DIALECT.kinds()["arraypattern"]  # TypeScript5's `Term` lists it
with raises(ValueError, match="TypeScript Patterns has no kind 'call'"):
    TSP.DIALECT.make(F.Form("call"))
assert F.form(term(3)) == F.Form("literal", {"value": 3})
assert F.form(term(E.let_("x", None, None))) == F.Form("let", {"name": "x"}, (None, None))  # empty subterms are None
assert F.form(term(lambda u: u.call(lambda c: c))) == F.Form("call")
with raises(ValueError, match="Patterns has no kind 'constant'"):
    E.DIALECT.make(F.Form("constant"))
with raises(ValueError, match="a literal has no attribute 'name'"):
    term(E.DIALECT.make(F.Form("literal", {"name": "x"})))
with raises(ValueError, match="a let takes 2 arguments, got 1"):
    term(E.DIALECT.make(F.Form("let", {"name": "x"}, (basic,))))
with raises(ValueError, match="a quantifier takes 2 arguments, got 3"):
    term(E.DIALECT.make(F.Form("quantifier", {"name": "x", "quantifier": "all"}, (1, 2, 3))))
spelled = term(lambda u: u.literal(lambda r: r.text("08").problems(["leading zeros"])))  # a container of scalars too
assert F.same(term(E.DIALECT.make(F.form(spelled))), spelled)
made = term(E.DIALECT.make(F.Form("call", {"callable": "f"}, (E.variable("g"), 1, 2))))  # specs: writers, natives
assert [F.kind_of(a)[0] for a in F.form(made).arguments] == ["variable", "literal", "literal"]
with raises(ValueError, match="a term holds no dialect"):
    F.kind_of(term(lambda u: u))  # nothing written: no branch of Term is selected
with store.transaction():  # a value of a dialect's `Term`, alone
    alone = store.Expressions.Patterns.Expressions.Term(lambda u: u)
assert F.is_term(alone) and F.dialect_of(alone) == E.DIALECT.union
with raises(ValueError, match="a term holds no kind"):
    F.kind_of(alone)
with raises(ValueError, match="a call takes at least 1 arguments, got 0"):  # its function, then its arguments
    term(P.DIALECT.make(F.Form(P.CALL)), P.DIALECT)
made = term(P.DIALECT.make(F.Form(P.CALL, {}, (P.name("f"), P.constant(1), 2))), P.DIALECT)  # a native: Python's constant
assert F.name_of(F.form(made).arguments[0]) == "f" and [F.form(a) for a in F.form(made).arguments[1:]] == [
    F.Form(P.LITERAL, {"value": 1}), F.Form(P.LITERAL, {"value": 2})]
assert F.kind_of(term(1, M.DIALECT))[0] == "literal" and F.kind_of(term(True, C.DIALECT))[0] == "literal"  # its literal kind
with raises(TypeError, match="expected a term, a writer or a native value, got [1]"):  # no literal kind holds these
    E.DIALECT.spec([1])
with raises(TypeError, match="an Excel literal cannot hold a bytes"):
    X.DIALECT.spec(b"x")
# A form and a part print as Python writes them, in messages (both languages alike)
assert repr(F.Form("call", {"callable": "add"}, ("x",))) == "Form(kind='call', attributes={'callable': 'add'}, arguments=('x',))"
assert repr(F.Part("keyword", {"arg": "x"}, 1)) == "Part(kind='keyword', attributes={'arg': 'x'}, size=1)"

<a id="FRM-03"></a>
## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
shared = E.variable("x")
pair = term(E.operation("f", shared, shared))
assert [F.form(e).kind for e in F.walk(pair)] == ["call", "variable", "variable"]  # a nested value: each visited
assert [F.form(e).attributes for e in F.walk(term(basic))][:3] == [{"name": "a"}, {"callable": "get"}, {"name": "person"}]
assert F.fold(pair, lambda node, results: 1 + sum(results)) == 3
calls = []
F.fold(pair, lambda node, results: calls.append(node))
assert len(calls) == 3  # once per term
assert F.fold(term(E.let_("x", None, 1)), lambda node, results: results) == [None, []]
for a, b, expected in [(1, 1, True), (1, 1.0, False), (math.nan, math.nan, True), (-0.0, 0.0, False),
                       (0.0, 0.0, True), ("x", "x", True), (b"x", b"x", True), (b"x", b"y", False)]:
    assert F.same(term(a), term(b)) is expected, (a, b)
assert F.same(term(basic), term(E.DIALECT.make(F.form(term(basic))))) and F.same(term(E.let_("x", None, None)), term(E.let_("x", None, None)))
assert not F.same(term(E.let_("x", None, None)), term(E.let_("x", 1, 2))) and not F.same(term(1), term(E.variable("x")))
assert not F.same(term(E.operation("f", 1)), term(E.operation("f", 1, 2)))
assert not F.same(term(E.operation("f")), term(lambda u: u.call(lambda c: c)))  # attributes differ

<a id="FRM-04"></a>
## FRM-04 · Every dialect's install adds its union of its kinds to `Term`, and its terms round-trip

In [ ]:
for dialect in DIALECTS:
    assert dialect.tag in {b.name for b in Term.branches}
    sample = samples[dialect.name()]
    for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
        back = decode(Term, encode(Term, sample))
        assert F.same(back, sample) and dialect.validate(term(back, dialect), bound={"person"}) == []  # held, as validation reads its store
# The framework reads a kind's subterms, variadic arguments and attributes from its schema: a subterm's schema is Term, or a
# dialect's union.
of = {d.name(): {b.name: b.schema for b in store.registered(d.union).branches} for d in DIALECTS}
kinds = of["Patterns"]
assert F._parts(kinds["let"]) == (("value", "body"), (), ("name",))
assert F._parts(kinds["call"]) == ((), ("arguments",), ("callable",))
assert F._parts(kinds["quantifier"]) == (("value", "body"), (), ("name", "quantifier"))  # a binder: Let's first
assert F._parts(kinds["literal"]) == ((), (), ("value", "text", "problems"))
assert F._parts(kinds["list"]) == ((), ("elements",), ()) and F._parts(kinds["mapping"]) == ((), ("entries",), ())
assert F._parts(kinds["error"]) == ((), ("terms",), ("text",)) and F._parts(kinds["missing"]) == ((), (), ("kind",))
assert F._members(kinds["mapping"], "entries") == ("key", "value") and F._members(kinds["list"], "elements") == ()
for dialect in DIALECTS[1:]:
    for tag in dialect.kinds():  # a dialect's own kinds are its schemas, named by it
        assert of[dialect.name()][tag].name.startswith((f"Expressions.{dialect.path}.", "Expressions.Framework.")) or (
            dialect is TST.DIALECT and tag == "identifier"), tag  # but TypeScript Types' identifier, TypeScript5's, read so
assert [of["Python3"][tag].name for tag in P.DIALECT.kinds()][-10:] == [
    "Expressions.Python.Python3.Compare", "Expressions.Python.Python3.BoolOp", "Expressions.Python.Python3.NamedExpr", "Expressions.Python.Python3.IfExp",
    "Expressions.Python.Python3.Lambda", "Expressions.Framework.Let", "Expressions.Python.Python3.Import", "Expressions.Python.Python3.ImportFrom",
    "Expressions.Framework.Error", "Expressions.Framework.Missing"]
assert of["Python3"][P.LITERAL].refines is store.registered("Expressions.Framework.Literal")  # the framework's, refined
assert F._parts(of["Python3"][P.CALL]) == (("func",), ("args",), ("keywords",))  # a subterm, variadic arguments, then helpers
assert F._helpers(of["Python3"][P.CALL]) == ("keywords",) and F._helpers(of["Python3"][P.BINOP]) == ()
assert F._parts(of["Python3"][P.GENERATOREXP]) == (("value", "body"), (), ("name", "generators"))  # a binder: Let's value empty
assert F._parts(of["Python3"][P.DICT]) == ((), ("items",), ()) and F._members(of["Python3"][P.DICT], "items") == ("key", "value")
assert F._parts(of["MATLAB"]["binary"]) == (("left", "right"), (), ("operator",))
assert F._parts(of["Excel"]["cell"]) == ((), (), ("address", "sheet", "book"))
assert of["Python3"][P.CALL].properties["func"].schema is store.registered(P.DIALECT.union)  # its own union
assert of["C"]["binaryexpression"].properties["left"].schema is store.registered(C.DIALECT.union)  # C's kinds hold C's terms
assert of["Cpp"]["binaryexpression"].properties["left"].schema is store.registered(CPP.DIALECT.union)  # and C++'s, C++'s
# A kind that binds names refines the framework's `Let` (a binder): SystemVerilog's array method with a `with` clause, its
# call Let's value and its `with` expression Let's body, each redeclared to hold its dialect's terms.
LET, WITH = store.registered("Expressions.Framework.Let"), of["SystemVerilog"]["arraymethodwithexpression"]
assert WITH.refines is LET and F.is_binder(WITH) and F.is_binder(LET) and not F.is_binder(of["SystemVerilog"]["callexpression"])
assert not F.is_binder(store.registered("Schemas.OfInt")) and F._parts(WITH) == (("value", "body"), (), ("name",))
assert WITH.properties["value"].schema is WITH.properties["body"].schema is store.registered(SV.DIALECT.union)
# Each dialect's holder, the root of an expression tree: a reference object refining the framework's Expression, its
# root narrowed to the dialect's Term. It validates, holds a term of its dialect, round-trips, prints (FRM-11), and HasRoot,
# which links the framework's Expression, links it.
Expression = store.registered("Expressions.Framework.Expression")
holders = {}
for dialect in DIALECTS:
    Holder = store.registered(f"Expressions.{dialect.path}.Expression")
    assert Holder.ref and Holder.refines is Expression and Holder.validate() == [] and F.is_expression(Holder)
    assert Holder.properties["root"].schema is store.registered(dialect.union)  # narrowed from Expression's Term
    sample = samples[dialect.name()]
    with store.transaction():
        held = functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression(  # C's and C++'s name their ABI
            lambda e: e.root(dialect.spec(sample)).abi([AMD64_]) if dialect.tag in ("c", "cpp") else e.root(dialect.spec(sample)))
        linked = store.Schemas.OfExpression(lambda s: s.name(f"frm.{dialect.name()}").expression(
            lambda a: a.insert(lambda e: e.expression(held))))
    holders[dialect] = held
    assert F.same(F.term(held), sample) and F.dialect_of(F.term(held)) == dialect.union
    assert Validators.Validate(store)(Holder, held) == [] and dialect.validate(held, bound={"person"}) == []
    assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, held)).root, held.root)
    assert [F.same(entry.expression, held) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
# Expression names the ABI its terms are read under, one at most, through its adjacency abi (HasAbi), which every
# dialect's holder inherits: C's and C++'s read the types their standards leave to the implementation from it
HasAbi = store.registered("Expressions.Framework.HasAbi")
assert list(Expression.adjacencies) == ["abi"] and Expression.adjacencies["abi"].relation is HasAbi and Expression.adjacencies["abi"].me == "expression"
assert list(HasAbi.entry.links) == ["expression", "abi"] and HasAbi.entry.links["abi"].schema is Dialects
assert list(store.registered("Expressions.C.Expressions.Expression").adjacencies) == ["abi"]
assert [entry.abi for entry in holders[C.DIALECT].abi] == [AMD64_] and list(holders[E.DIALECT].abi) == []
with raises(Exception, match="'quantifier' is not a member of a union value"):  # its root holds its dialect's terms alone
    with store.transaction():
        store.Expressions.Python.Python3.Expression(lambda e: e.root(E.list_(1).all("p", True)))

<a id="FRM-05"></a>
## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
assert [d.validate(samples[d.name()], bound={"person"}, core=True) for d in DIALECTS] == [[]] * len(DIALECTS)
assert E.DIALECT.validate(term(E.operation("add", 1))) == ["add takes 2 arguments, got 1"]
assert E.DIALECT.validate(term(E.operation("not", E.variable("x")))) == ["argument 0: variable 'x' is not bound"]
assert E.DIALECT.validate(term(E.mapping((E.variable("k"), 1), ("b", E.list_(E.variable("x")))))) == [
    "key 0: variable 'k' is not bound", "value 1: argument 0: variable 'x' is not bound"]  # a mapping's terms, by entry
assert E.DIALECT.validate(term(lambda u: u.mapping(lambda m: m.entries([lambda a: a.as_object(lambda o: o.key(E.literal(1)))])))) == [
    "a mapping needs a value 0"]
assert E.DIALECT.validate(term(E.let_("a", E.variable("a"), E.variable("a")))) == ["value: variable 'a' is not bound"]
assert E.DIALECT.validate(term(lambda u: u.let(lambda r: r.name("x").value(E.literal(1))))) == ["a let needs a body"]
for spec, bound, problems in [
    (P.subscript(P.name("t"), "x"), (), ["value: variable 't' is not bound"]),
    (P.call("f", 1), (), ["func: variable 'f' is not bound"]),  # a subterm, then variadic arguments
    (P.call(P.name("f"), P.name("x")), {"f"}, ["argument 0: variable 'x' is not bound"]),
    (P.call(P.name("f"), P.keyword("k", P.name("x"))), {"f"}, ["keywords 0 value: variable 'x' is not bound"]),  # a helper's
    (lambda u: getattr(u, P.CALL)(lambda r: r.func(P.name("f")).keywords([lambda a: a.as_object(lambda o: o.arg("k"))])),
     {"f"}, ["a call needs a keywords 0 value"]),
    (P.listcomp(P.name("y"), P.comprehension("y", P.name("y"))), (), ["generators 0 iter: variable 'y' is not bound"]),
    (P.lambda_(P.name("a"), ("a", P.arg("b", P.name("a")))), (), ["args args 1 default_value: variable 'a' is not bound"]),
    (P.subscript(P.name("t"), P.slice_()), {"t"}, []),  # a kind's optional subterms may be empty
    (P.import_("numpy", P.name("np"), alias="np"), (), []),  # an import binds a name; its alias is optional
    (P.import_("numpy", P.name("np")), (), ["body: variable 'np' is not bound"]),
    (lambda u: getattr(u, P.IMPORT)(lambda r: r.module("numpy")), (), ["an import needs a body"]),
    (P.name("not a name"), (), ["a name must be an identifier, got 'not a name'"]),  # a kind's own check
]:
    assert P.DIALECT.validate(term(spec, P.DIALECT), bound) == problems
assert X.DIALECT.validate(term(X.cell("A1"), X.DIALECT)) == [] and X.DIALECT.validate(term(X.cell("B2", "Sheet1"), X.DIALECT), core=True) == []  # cells need no binding
assert M.DIALECT.validate(term(lambda u: u.field(lambda r: r), M.DIALECT)) == ["a field needs a name", "a field needs a value"]
assert M.DIALECT.validate(term(lambda u: u.binary(lambda r: r.operator("&&").left(M.constant(1))), M.DIALECT)) == ["a binary needs a right"]
assert M.DIALECT.validate(term(M.binary("**", 1, 2), M.DIALECT)) == [] and M.DIALECT.validate(term(M.binary("**", 1, 2), M.DIALECT), core=True) == [
    "'**' is not a core operation"]
assert M.DIALECT.validate(term(M.field(M.identifier("s"), "any name"), M.DIALECT), {"s"}, core=True) == []  # a field's vocabulary is open
assert X.DIALECT.validate(term(X.function("NOT", True, False), X.DIALECT)) == ["NOT takes 1 arguments, got 2"]
assert X.DIALECT.validate(term(X.function("AND", True), X.DIALECT)) == []  # AND: any number
assert X.DIALECT.validate(term(X.let_("a", X.name("a"), X.name("a")), X.DIALECT)) == ["value: variable 'a' is not bound"]
assert X.DIALECT.validate(term(lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(b"x"))), X.DIALECT)) == [
    "an Excel literal cannot hold a bytes"]  # the dialect's own check of the framework's literal
assert M.DIALECT.validate(term(lambda u: u.call(lambda r: r.callable(7)), M.DIALECT)) == ["a call's callable must be a str, got int"]

<a id="FRM-06"></a>
## FRM-06 · Domains and signatures

In [ ]:
Int, Float, Str, Odd = D.OfTypes("int", int), D.OfTypes("float", float), D.OfTypes("str", str), D.OfValues("odd", lambda v: v % 2 == 1)
Number = D.OfUnion(Int, Float)
assert Int.contains(1) and not Int.contains(True) and not Int.contains(1.0) and repr(Int) == "int"
assert D.OfTypes("bool", bool).contains(True) and not D.OfTypes("bool", bool).contains(1)
assert Str.contains("a") and not Str.contains(b"a")
assert Number.name() == "int | float" and repr(Number) == "int | float" and Number.contains(1.5) and not Number.contains("1")
assert Number.includes(Int) and Number.includes(D.OfUnion(Float, Int)) and not Int.includes(Number)
assert D.OfTypes("num", int, float).includes(Number) and not Number.includes(Str) and not Number.includes(D.Anything)
assert Odd.contains(3) and Odd.includes(Odd) and not Odd.includes(Int) and Odd.includes(D.OfUnion(Odd)) and repr(Odd) == "odd"
assert D.Anything.includes(Number) and D.Anything.contains(None) and repr(D.Anything) == "any" and D.Anything.name() == "any"
assert D.overlaps(Int, D.Anything) and D.overlaps(Number, Int) and not D.overlaps(Int, Str)
assert D.OfTypes("list", list).contains([1]) and not D.OfTypes("list", list).contains("x")  # a class, by instance
f = D.Function((Int, Str), Float)
assert f.arity() == 2 and f.result([Int, Str]) is Float and f.result([D.Anything, Str]) is Float
assert f.result([Str, Str]) is None and f.result([Int]) is None and f.describe() == "(int, str) -> float"
assert f.exact([Int, Str]) and not f.exact([D.Anything, Str])
same = D.Same(2, (Int, Float))
assert same.result([Int, Int]) is Int and same.result([Int, Float]) is None and same.result([Int]) is None
assert same.result([D.Anything, D.Anything]).name() == "int | float" and same.result([D.Anything, Float]) is Float
assert same.describe() == "(T, T) -> T for T in int | float" and D.Same(1, (Int,), Str).describe() == "(T) -> str for T in int"
assert D.Same(2, (Int, Float), Str).result([D.Anything, D.Anything]) is Str  # one result, however many match
assert D.Opaque().arity() == -1 and D.Opaque(Int).result([Str, Str, Str]) is Int and D.Opaque().describe() == "(...) -> any"
either = D.Either(2)
assert either.arity() == 2 and either.result([Int, Int]) is Int and either.result([Int, Str]).name() == "int | str"
assert either.result([Int]) is None and either.describe() == "(T, T) -> T"
any_number = D.Either(-1)  # Python's `and` of any number of values
assert any_number.result([Int, Str, Int]).name() == "int | str" and any_number.result([]) is None and any_number.describe() == "(T, ...) -> T"
assert same.exact([Int, Int]) and not same.exact([Int, Float]) and not same.exact([Int])
over = D.Overloaded(D.Same(2, (Int,)), D.Function((Number, Number), Float))
assert over.arity() == 2 and over.result([Int, Int]) is Int and over.result([Int, Float]) is Float
assert over.result([D.Anything, Int]).name() == "int | float" and over.result([Str, Str]) is None
assert over.describe() == "(T, T) -> T for T in int | (int | float, int | float) -> float"

<a id="FRM-07"></a>
## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:


def infer(spec, environment=None):
    return E.DIALECT.infer(term(spec), environment)


assert infer(basic, {"person": BD.Object}) is BD.Bool
assert infer(E.literal(1).add(2)) is BD.Int and infer(E.literal(1.5).neg()) is BD.Float
assert infer(person.age, {"person": BD.Object}) is D.Anything
assert infer(person.age.add(1), {"person": BD.Object}) is BD.Int  # only int + int applies
assert M.DIALECT.infer(term(M.call("dist", 1, 2), M.DIALECT)) is D.Anything  # an extension: nothing is known of it
with raises(ValueError, match="cannot infer the domain of an invalid expression: 'contains' is not an operation Patterns defines"):
    infer(E.operation("contains", 1, 2))  # Patterns' operations are the ones it defines
assert infer(E.list_(1, 2)).name() == "list(int)" and infer(E.list_(1, "a")).name() == "list(int | str)"
assert infer(E.list_()).name() == "list(any)" and infer(E.list_(1, 2).count()) is BD.Int
assert infer(E.mapping(("a", 1.5))).name() == "keyed(str, float)" and infer(E.mapping()).name() == "keyed(any, any)"
assert infer(E.list_(1, 2).item(0)) is BD.Int and infer(E.list_(1, 2).all("x", E.variable("x").gt(0))) is BD.Bool
assert BD.LIST.describe() == "(T, ...) -> list(T)" and BD.MAPPING.describe() == "(K, T, ...) -> keyed(K, T)" and BD.LIST.arity() == -1
assert infer(E.let_("a", 1, E.let_("b", "s", E.variable("a")))) is BD.Int
with raises(TypeError, match="add cannot take (int, float); it takes (T, T) -> T for T in int | float"):
    infer(E.literal(1).add(1.5))
with raises(TypeError, match="has cannot take (int, str); it takes (object | record, str) -> bool"):
    infer(E.literal(1).has("x"))
with raises(ValueError, match="cannot infer the domain of an invalid expression: variable 'person' is not bound"):
    infer(person)
assert BD.of(term(1)) is BD.Object and BD.of(b"") is BD.Bytes  # a term value is an object
with raises(TypeError, match="a list is not a value of the Patterns dialect"):
    BD.of([])
assert M.DIALECT.infer(samples["MATLAB"], {"person": D.Anything}).name() == "logical"
assert X.DIALECT.infer(samples["Excel"], {"person": D.Anything}).name() == "logical"
assert L.DIALECT.infer(samples["LaTeX"], {"person": D.Anything}).name() == "truth"
assert P.DIALECT.infer(samples["Python3"], {"person": D.Anything}) is D.Anything  # a call gives what is not known
assert TS.DIALECT.infer(samples["TypeScript5"], {"person": D.Anything}) is D.Anything  # nothing is known but a literal's
assert P.DIALECT.infer(term(P.boolop("and", 1, "a"), P.DIALECT)).name() == "int | str"  # Python's and gives an operand
assert P.DIALECT.infer(term(P.import_("math", P.binop("+", 1, True)), P.DIALECT)).name() == "int"

<a id="FRM-08"></a>
## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
def simplifier(rules, dialect=E.DIALECT):
    """A simplifier by `rules`, its literals a dialect's natives'."""
    return FV.Simplifier(rules, literal=FV.natives(dialect))


simplify = simplifier(FV.Interpreter(E.DIALECT, {"call": {}, "quantifier": {}}))
with raises(TypeError, match="not a term: 3"):
    simplify(3)
with raises(ValueError, match="a literal needs a value"):
    simplify(term(E.literal(None)))
unbound = simplify(term(E.variable("x")))  # not bound: it stays
assert unbound.free == {"x"} and not unbound.literal and unbound.residues == ()
kept = simplify(term(E.literal(True).and_(False)))  # no rule for and: it stays, named
assert [(r.dialect, r.kind, r.what, r.category) for r in kept.residues] == [("Patterns", "call", "call 'and'", "a")]
assert kept.residues[0].reason == "Patterns call 'and' is not simplified" and not kept.literal
typed = simplifier(FV.Interpreter(C.DIALECT, {}), C.DIALECT)(term(C.literal(5, "u"), C.DIALECT))  # a kind's own domain: C's type
assert [r.reason for r in typed.residues] == ["C literal of unsigned int is not simplified"]
with raises(TypeError, match="not takes 1 arguments, got 2"):
    simplifier(FV.Interpreter(E.DIALECT, {"call": {"not": None}}))(term(E.operation("not", 1, 2)))
with raises(ValueError, match="a let needs a body"):
    V.OfAny(term(E.let_("a", 1, None)))
with raises(ValueError, match="a quantifier needs a value"):
    V.OfAny(term(E.quantifier("all", "x", None, True)))
assert simplifier(FV.Interpreter(E.DIALECT, {"call": {}}, extension=lambda operator, thunks, node, scope: len(operator)))(
    term(E.operation("dist", 1, 2))).value == 4  # operators outside the vocabulary go to the extension

scope = SY.Variables({"x": 1})
x = term(E.variable("x"))
assert isinstance(scope, SY.Scope) and scope.lookup(x) == 1
inner = scope.bind("x", 2)
assert inner.lookup(x) == 2 and scope.lookup(x) == 1  # bind makes a new scope
with raises(SY.Unbound, match="y"):  # which a simplifier catches, leaving the reference
    scope.lookup(term(E.variable("y")))
with raises(SY.Unbound, match="x"):
    scope.bind("x", SY.UNKNOWN).lookup(x)  # a name hidden
assert V.OfAny(x, inner).value == 2  # a simplifier takes a scope as well as a mapping

<a id="FRM-09"></a>
## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
assert SY.free(term(basic), E.DIALECT) == {"person"} and SY.free(term(1), E.DIALECT) == set()
assert SY.free(term(E.let_("a", E.variable("a"), E.variable("a"))), E.DIALECT) == {"a"}  # the name is bound in the body only
assert SY.free(term(E.let_("a", None, E.variable("b"))), E.DIALECT) == {"b"}  # empty subterms are skipped
assert SY.free(term(E.quantifier("all", "x", E.variable("xs"), E.variable("x").gt(E.variable("y")))), E.DIALECT) == {"xs", "y"}
assert SY.imports(term(basic), E.DIALECT) == []
assert FE.NameError is NameError and FE.ImportError is ImportError  # Python's own exceptions, by the framework's names
assert FE.OverflowError is OverflowError and FE.ZeroDivisionError is ZeroDivisionError and FE.IndexError is IndexError
assert SY.free(samples["Python3"], P.DIALECT) == {"person"}  # np, f and a are bound by imports and the let
assert SY.free(samples["TypeScript5"], TS.DIALECT) == {"person"}  # p by its arrow; properties and keys name, not refer
assert SY.free(term(P.call("hasattr", P.name("x"), "k"), P.DIALECT), P.DIALECT) == {"x"}  # builtins are ambient
assert SY.free(term(X.infix("+", X.cell("A1"), X.name("r")), X.DIALECT), X.DIALECT) == {"r"}  # cells are resolved by the workbook, not bound
assert SY.free(term(M.import_("pkg.*", M.identifier("x")), M.DIALECT), M.DIALECT) == {"x"}  # MATLAB's imports bind functions, not variables
assert [F.kind_of(i)[0] for i in SY.imports(samples["Python3"], P.DIALECT)] == ["import", "importfrom"]
assert [F.name_of(i) for i in SY.imports(samples["MATLAB"], M.DIALECT)] == ["pkg.*"]

<a id="FRM-10"></a>
## FRM-10 · A dialect may extend another: its kinds are the base's and its own, added to its union by schema extension

In [ ]:


def install(txn):
    """An extension of Patterns, added to its union: `twice(x)`, x + x; `import(module, body)`, an import binding the
    module's name within its body; `cell`, a reference the scope resolves without a binding; and `apply(function,
    arguments)`, a subterm and variadic arguments. And `Bare`, a dialect of its own: its `Term`, `Expressions.Bare.Term`,
    the framework's literal and the import, the branch `bare` of Term."""
    S, Framework = txn.Schemas, txn.Expressions.Framework
    return txn.uid("frm.Ext/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Term": S.OfUnion, "Patterns": S.OfUnion, "Bare": S.OfUnion, "Twice": S.OfObject,
                  "Import": S.OfObject, "Cell": S.OfObject, "Apply": S.OfObject})
        .spec([
            lambda _: _.Twice.name("frm.Twice").properties({"operand": _.Term, "domain": S.OfStr}),
            lambda _: _.Import.name("frm.Import").properties({"name": S.OfStr, "body": _.Term, "domain": S.OfStr}),
            lambda _: _.Cell.name("frm.Cell").properties({"name": S.OfStr, "domain": S.OfStr}),
            lambda _: _.Apply.name("frm.Apply").properties({
                "function": _.Term, "arguments": S.OfContainer(lambda c: c.index([S.OfInt]).element([_.Term])), "domain": S.OfStr}),
            lambda _: _.Patterns.name("Expressions.Patterns.Expressions.Term").branches([  # Patterns' union, extended
                lambda t: t.name("twice").schema([_.Twice]), lambda t: t.name("import").schema([_.Import]),
                lambda t: t.name("cell").schema([_.Cell]), lambda t: t.name("apply").schema([_.Apply])]),
            lambda _: _.Bare.name("Expressions.Bare.Term").branches([
                lambda t: t.name("literal").schema([Framework.Literal]), lambda t: t.name("import").schema([_.Import])]),
            lambda _: _.Term.name("Expressions.Framework.Term").branches([lambda t: t.name("bare").schema([_.Bare])]),
        ])]))])


store.transaction(install)
Ext = F.Dialect("Ext", {
    "twice": F.Kind(F.APPLICATION, signature=D.Function((BD.Int,), BD.Int)),  # an open vocabulary: its tag names it
    "import": F.Kind(F.IMPORT, required=("name",), names=lambda attributes: (attributes["name"],)),
    "cell": F.Kind(F.REFERENCE, required=("name",), lexical=False),
    "apply": F.Kind(F.APPLICATION, signature=D.Opaque()),
}, domain_of=BD.of, extends=E.DIALECT)
assert list(Ext.kinds()) == [*E.DIALECT.kinds(), "twice", "import", "cell", "apply"] and Ext.base is E.DIALECT and E.DIALECT.base is None
assert (Ext.tag, Ext.union) == (E.DIALECT.tag, E.DIALECT.union)  # its terms are its base's union's
with raises(ValueError, match="an apply takes at least 1 arguments, got 0"):
    term(Ext.make(F.Form("apply")), Ext)
twice = lambda operand: E.Writer(lambda u: u.twice(lambda r: r.operand(Ext.spec(operand))))  # noqa: E731
import_ = lambda name, body, dialect=Ext: F.Writer(  # noqa: E731
    lambda u: getattr(u, "import")(lambda r: r.name(name).body(dialect.spec(body))), dialect.tag)
cell = lambda name: F.Writer(lambda u: u.cell(lambda r: r.name(name)), Ext.tag)  # noqa: E731
x = E.variable("x")
mixed = term(x.add(twice(x)), Ext)  # a writer of the base holds the extension's term
assert Ext.validate(mixed, bound={"x"}) == [] and F.kind_of(F.form(mixed).arguments[1])[0] == "twice"
assert E.DIALECT.validate(mixed, bound={"x"}) == ["argument 1: Patterns has no kind 'twice'"]  # the base reads only its own
with raises(ValueError, match="Patterns has no kind 'twice'"):
    V.OfAny(mixed, {"x": 1}).value


class Workbook(SY.Variables):
    """A scope with modules to import and cells to read."""

    def __init__(self, variables=None, cells=None, modules=None):
        super().__init__(variables)
        self.cells, self.modules = dict(cells or {}), dict(modules or {})

    def lookup(self, reference):
        if F.kind_of(reference)[0] == "cell":
            return self.cells[F.name_of(reference)]
        return super().lookup(reference)

    def enter(self, declaration):
        name = F.name_of(declaration)
        if name not in self.modules:
            raise SY.Unbound(name)
        return self.bind(name, self.modules[name])


interpreter = FV.Interpreter(Ext, {"call": V.OPERATIONS, "quantifier": V.QUANTIFIERS,
                                   "twice": lambda thunks, node, scope: thunks[0]() * 2}, scope=Workbook)
extended = FV.Simplifier(interpreter, literal=lambda value: E.literal(value) if type(value) is int else None)
assert extended(mixed, {"x": 3}).value == 9
program = term(import_("m", E.variable("m").get("k").add(cell("A1"))), Ext)
unknown = extended(program, Workbook(cells={"A1": 1}, modules={"m": term(E.literal(1))}))  # no property k: unknown
assert unknown.value is None and not unknown.literal
assert Ext.validate(program) == [] and SY.free(program, Ext) == set()  # the import binds m; a cell needs no binding
assert [F.name_of(i) for i in SY.imports(program, Ext)] == ["m"]
assert Ext.infer(program).name() == "int | float" and Ext.infer(mixed, {"x": BD.Int}) is BD.Int
with raises(SY.Unbound, match="import"):  # Variables has no imports: an import stays
    SY.Variables().enter(program)
with raises(ValueError, match="an import needs a body"):
    extended(term(lambda u: getattr(u, "import")(lambda r: r.name("m")), Ext))
assert Ext.validate(term(lambda u: getattr(u, "import")(lambda r: r.body(E.variable("m"))), Ext)) == ["an import needs a name", "body: variable 'm' is not bound"]
# Simplifying an extension's terms: an import the scope does not provide stays, binding its names within its body.
needs_y = term(import_("m", E.variable("m").get("k").add(E.variable("y"))), Ext)
assert F.same(term(extended(needs_y, {"x": 1}).residual, Ext), needs_y)
assert F.same(term(extended(term(twice(x).add(twice(E.variable("y"))), Ext), {"x": 2}).residual, Ext), term(E.literal(4).add(twice(E.variable("y"))), Ext))
provided = extended(needs_y, Workbook({"y": 2}, modules={"m": term(E.literal(1))}))  # provided: entered, its body simplified
assert provided.value is None and F.same(term(provided.residual, Ext), term(import_("m", E.variable("m").get("k").add(2)), Ext))
# A dialect's kinds need no checks, domains or names of their own.
Bare = F.Dialect("Bare", {"literal": F.Kind(F.LITERAL), "import": F.Kind(F.IMPORT)}, domain_of=BD.of)
three = lambda u: u.literal(lambda r: r.value(lambda p: p.as_scalar(3)))  # noqa: E731: Bare has no writers
bare = term(import_("m", three, Bare), Bare)
assert Bare.validate(bare) == [] and SY.free(bare, Bare) == set() and Bare.infer(bare) is BD.Int
assert FV.Simplifier(FV.Interpreter(Bare, {}), literal=lambda value: None)(term(three, Bare)).value == 3  # kept as written
with raises(ValueError, match="Clash would declare 'literal', which Patterns has"):
    F.Dialect("Clash", {"literal": F.Kind(F.APPLICATION)}, domain_of=BD.of, extends=E.DIALECT)
with raises(ValueError, match="Clash would declare 'error', which the framework has"):  # every dialect reads invalid syntax
    F.Dialect("Clash", {"error": F.Kind(F.APPLICATION)}, domain_of=BD.of)

<a id="FRM-11"></a>
## FRM-11 · Invalid syntax, `error` and `missing`: every dialect's `Term`, `Type` and `Pattern` hold them, validation reports them, evaluation refuses them by name, and printers write them as written

In [ ]:
from mbse.Expressions.C.Expressions import Text as CText
from mbse.Expressions.Cpp.Expressions import Text as CPPText
from mbse.Expressions.Excel.Expressions import Text as XText
from mbse.Expressions.LaTeX.Expressions import Text as LText
from mbse.Expressions.MATLAB.Expressions import Text as MText
from mbse.Expressions.Python.Python3 import Text as PText
from mbse.Expressions.SystemVerilog.Expressions import Text as SVText
from mbse.Expressions.TypeScript.TypeScript5 import Text as TSText

PRINTERS = {P.DIALECT: PText.ToText, M.DIALECT: MText.ToText, X.DIALECT: XText.ToText, L.DIALECT: LText.ToText,
            C.DIALECT: CText.ToText, CPP.DIALECT: CPPText.ToText, SV.DIALECT: SVText.ToText,
            **{package.Expressions.DIALECT: package.Text.ToText for package in (SVTypes, SVPatterns, SVAssertions, SVConstraints,
                                                                               SVPreprocessor)},  # its family's, one printer
            TS.DIALECT: TSText.ToText, TST.DIALECT: TypeScriptTypes.Text.ToText,
            TSP.DIALECT: TypeScriptPatterns.Text.ToText, PT: PythonTypes.Text.ToText, PP: PythonPatterns.Text.ToText,
            PR: PythonRegExp.Text.ToText, TR: TypeScriptRegExp.Text.ToText}  # Patterns has none
assert all(package.Text.ToText is SVText.ToText for package in (SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor))
assert TypeScriptTypes.Text.ToText is TypeScriptPatterns.Text.ToText is TSText.ToText  # its family's, one printer
assert all(write(holders[dialect]) == write(samples[dialect.name()]) for dialect, write in PRINTERS.items())  # FRM-04's
assert TSText.ToText(samples["TypeScript5"]) == (  # every kind, a body of statements written `{}`
    '({ a: [person,, ...person], d: { person }, person = 1 } = person), async function* g<T>(x: T = this.y) {}, as'
    'ync <U,>(p: U, ...r: U[]) => await import.meta.resolve(p), @person class K<T> extends person.Base<T> implements person {}, new'
    ' person.C<number>(...person), person`a${person}b`, `x${person}`, person++, typeof person, #p in person, person'
    ' ?? null, person ? /x/g : 1n, yield* person, person as string, person satisfies object, <any>person, person!, '
    'person<string>, [1,, ...person], { m(): void, ...person, [person]: "s" }, <div id="x" {...person}>{}{person}<>'
    'text</></div>, <><br /></>, person?.a, super.x, import(person)')


def recovered(dialect):
    """A name used as an expression, as each dialect's grammar makes it: TypeScript5's identifier, C++'s idexpression
    holding its identifier, the framework's variable."""
    if dialect in (PR, TR):  # a character, each RegExp dialect's own: its Term lists no kind of its family's
        return lambda v: v.character(lambda w: w.value(97))
    if dialect in TS_FAMILY:  # in each of TypeScript's family's unions
        return lambda v: v.identifier(lambda w: w.name("a"))
    if dialect is CPP.DIALECT:
        return lambda v: v.idexpression(lambda w: w.name(lambda n: n.identifier(lambda i: i.spelling("a"))))
    if dialect is PP:  # a capture pattern, Python Patterns' own: its Term lists no Python3 kind
        return lambda v: v.matchas(lambda w: w.name("a"))
    if dialect is SVPP.DIALECT:  # a macro's usage, the Preprocessor's one kind yet
        return lambda v: v.macrousage(lambda w: w.name(lambda i: i.spelling("a")))
    if dialect in FAMILY:  # SystemVerilog's nameexpression, holding its identifier, in each of its family's unions
        return lambda v: v.nameexpression(lambda w: w.name(lambda n: n.identifier(lambda i: i.spelling("a"))))
    return lambda v: v.variable(lambda w: w.name("a"))


error = lambda dialect: lambda u: u.error(lambda r: r.text("(a +* 1)").terms([  # noqa: E731: what a parser skipped
    dialect.spec(recovered(dialect))]))  # and the terms it recovered, of its dialect
missing = lambda u: u.missing(lambda r: r.kind("expression"))  # noqa: E731: what it expected
for dialect in DIALECTS:
    skipped, expected = term(error(dialect), dialect), term(missing, dialect)
    assert F.form(skipped).kind == "error" and F.form(skipped).attributes == {"text": "(a +* 1)"}
    assert len(F.form(skipped).arguments) == 1 and F.form(expected) == F.Form("missing", {"kind": "expression"})
    assert dialect.validate(skipped) == ["invalid syntax '(a +* 1)'"] and dialect.validate(expected) == ["missing expression"]
    assert dialect.validate(term(lambda u: u.error(lambda r: r), dialect)) == ["invalid syntax"]
    assert dialect.validate(term(lambda u: u.missing(lambda r: r), dialect)) == ["missing a term"]
    assert F.same(term(dialect.make(F.form(skipped)), dialect), skipped)  # forms and make, as for every kind
    assert [dialect.kind(t)[2].role for t in (skipped, expected)] == [F.INVALID, F.INVALID]
    if dialect in SIMPLIFIERS:  # invalid syntax stays, named: it has no value of its own
        for invalid, kind in ((skipped, "error"), (expected, "missing")):
            first = SIMPLIFIERS[dialect](invalid).residues[0]  # then what it recovered, where no rule folds it
            assert (first.reason, first.category) == (f"{dialect.name()} {kind} is not simplified", "b")
    if dialect in PRINTERS:  # an error as the source it spans, a missing term as nothing
        assert PRINTERS[dialect](skipped) == "(a +* 1)" if dialect is not X.DIALECT else XText.ToText(skipped) == "=(a +* 1)"
        assert PRINTERS[dialect](expected) == ("=" if dialect is X.DIALECT else "")
        with raises(ValueError, match=f"{dialect.name()} error has no text to print"):
            PRINTERS[dialect](term(lambda u: u.error(lambda r: r), dialect))
assert PText.ToText(term(P.binop("+", P.name("a"), missing), P.DIALECT)) == "a + "  # within a term, as written
assert SY.free(term(error(E.DIALECT)), E.DIALECT) == {"a"}  # the terms recovered within an error are terms
with raises(ValueError, match="Python3 error has no MATLAB counterpart"):  # nothing translates invalid syntax
    Translators.between(P.DIALECT, M.DIALECT).forward(term(error(P.DIALECT), P.DIALECT))
with raises(ValueError, match="cannot infer the domain of an invalid expression: missing expression"):
    E.DIALECT.infer(term(missing))
# Every dialect's Type and Pattern hold invalid syntax too: validation reports it within a type or a target, printers
# write it as written, evaluation refuses it by name
for union in ("Python.Python3.Pattern", "TypeScript.Types.Term", "TypeScript.Patterns.Term", "C.Types.Term",
              "C.Expressions.Pattern", "Cpp.Types.Term", "Cpp.Expressions.Pattern"):
    assert INVALID <= {b.name for b in store.registered(f"Expressions.{union}").branches}, union
one = lambda v: v.literal(lambda li: li.value(lambda n: n.as_scalar(1)))  # noqa: E731: Python3's and C's
named = term(lambda u: u.namedexpr(lambda r: r.target(lambda t: t.error(lambda e: e.text("1"))).value(one)), P.DIALECT)  # (1 := 1)
assert P.DIALECT.validate(named) == ["target: invalid syntax '1'"] and PText.ToText(named) == "(1 := 1)"
generator = term(P.generatorexp(P.name("p"), P.comprehension(lambda t: t.missing(lambda m: m.kind("identifier")), P.list_(1))), P.DIALECT)
assert P.DIALECT.validate(generator, {"p"}) == ["generators 0 target: missing identifier"] and PText.ToText(generator) == "(p for  in [1])"
with raises(NotEvaluated, match="Python3 missing is not simplified"):
    PV.OfAny(generator).evaluate()
cast = term(lambda u: u.tsasexpression(lambda r: r.expression(ts_person).typeAnnotation(lambda t: t.missing(lambda m: m.kind("type")))), TS.DIALECT)
assert TS.DIALECT.validate(cast, {"person"}) == ["typeAnnotation: missing type"] and TSText.ToText(cast) == "person as "
arrow = term(lambda u: u.arrowfunctionexpression(lambda r: r.params([lambda p: p.error(lambda e: e.text("a b"))])), TS.DIALECT)
assert TS.DIALECT.validate(arrow) == ["argument 0: invalid syntax 'a b'"] and TSText.ToText(arrow) == "(a b) => {}"
for dialect, printer in [(C.DIALECT, CText.ToText), (CPP.DIALECT, CPPText.ToText)]:
    typed = term(lambda u: u.castexpression(lambda r: r.type(lambda t: t.error(lambda e: e.text("int int"))).operand(one)), dialect)
    assert dialect.validate(typed) == ["type: invalid syntax 'int int'"] and printer(typed) == "(int int)1"  # its type a Types term

<a id="FRM-12"></a>
## FRM-12 · A dialect reads only its own terms, and those of the dialects it reads: a term holds its dialect, a shared kind's subterm may be any dialect's term, and validation reports another's; C and C++ each define the kinds they share; SystemVerilog's six dialects read each other's

In [ ]:
foreign = term(P.let_("a", E.variable("a"), P.name("a")), P.DIALECT)  # a shared kind's subterm: Patterns' variable in it
assert F.dialect_of(F.form(foreign).arguments[0]) == E.DIALECT.union and F.dialect_of(foreign) == P.DIALECT.union
assert P.DIALECT.validate(foreign) == ["value: a Patterns variable is not a Python3 term"]
with raises(ValueError, match="a Patterns variable is not a Python3 term"):
    PV.OfAny(foreign).evaluate()
with raises(ValueError, match="an Excel call is not a MATLAB term"):
    MText.ToText(term(X.function("AND", True), X.DIALECT))
assert not F.same(term(E.variable("a")), term(P.name("a"), P.DIALECT))  # the same kind, in another dialect
own = term(P.binop("+", P.name("a"), 1), P.DIALECT)  # a dialect's own kind's subterm: its union's terms alone
assert [F.dialect_of(a) for a in F.form(own).arguments] == [P.DIALECT.union] * 2
assert F.form(own).arguments[0].as_schema().name == P.DIALECT.union  # one level: its kind alone
with raises(Exception, match="'cell' is not a member of a union value"):  # the schema refuses a kind its union lacks
    term(P.binop("+", X.cell("A1"), 1), P.DIALECT)
# C and C++: each its own kinds and its own Term (the standards that have each are data: FRM-13)
with raises(TypeError, match="expected a term, a writer or a native value, got b'x'"):  # no literal of C++ holds bytes
    CPP.DIALECT.spec(b"x")
assert C.EXTENSIONS == CPP.EXTENSIONS == {"statementexpression", "extensionexpression"}  # GNU's
assert CText.ToText(samples["C"]) == (  # every kind, a body of statements written `{}`
    '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, "a" "b", (person), _Generic(person, int: 1, d'
    'efault: 0), person++, sizeof(int), _Alignof(person), person += 1, .x[0] = person, (int[]){1}, ({}), __extension__ person}')
assert CPPText.ToText(samples["Cpp"]) == (
    '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, 10_km, "a" R"(b)", std::cout, this, (person), '
    '[=, &person](auto y) {}, (person + ...), requires (int z) { z + 1; }, int(person), person--, static_cast<int>(person'
    '), typeid(person), co_await person, sizeof(int), sizeof...(Ts), alignof(int), noexcept(person), new int{1}, delete['
    '] person, person = 0, throw, co_yield person, person..., .x = person, ({}), __extension__ person}')
assert CV.OfAny(term(C.binary("+", 1, 2), C.DIALECT)).evaluate() == CPV.OfAny(term(CPP.binary("+", 1, 2), CPP.DIALECT)).evaluate() == 3
with raises(Exception, match="'call' is not a member of a union value"):  # C's own kinds hold C's terms alone
    term(C.binary("+", E.variable("x").add(1), 1), C.DIALECT)
# SystemVerilog's dialects read each other's terms (`reads`): a cast's type is a Types term, read by SystemVerilog as
# Types reads it, and a dimension's bound an Expressions term, read by Types as SystemVerilog reads it; any other
# dialect's term is refused as before, a sublanguage's named by its family and its own name.
cast = term(SV.cast(sv_atom("int"), sv_person), SV.DIALECT)
typed = F.form(cast).arguments[0]
assert F.dialect_of(typed) == SVT.DIALECT.union and SV.DIALECT.kind(typed)[2] is SVT.DIALECT.kinds()["integeratomtype"]
assert SV.DIALECT.validate(cast, {"person"}) == [] and SVT.DIALECT.validate(samples["SystemVerilog Types"], {"person"}) == []
assert SV.DIALECT.validate(term(SV.cast(lambda t: t.integeratomtype(lambda a: a.keyword("word")), 1), SV.DIALECT)) == [
    "type: IntegerAtomType.keyword cannot be 'word'"]  # its checks, as Types reads it
assert C.DIALECT.validate(typed) == ["a SystemVerilog Types integeratomtype is not a C term"]
assert SV.DIALECT.validate(term(E.variable("a"))) == ["a Patterns variable is not a SystemVerilog term"]
with raises(ValueError, match="a SystemVerilog Types integeratomtype is not a Patterns term"):
    E.DIALECT.kind(typed)

<a id="FRM-13"></a>
## FRM-13 · Standards are data: each dialect covers its editions by designation, each found by its designation or its common names, and records the kinds it has

In [ ]:
# Each dialect's define writes the editions it covers, owned by their designations as their standards bodies write them,
# each holding its formal reference's parts and its common names: the tags it is found by, either kind of reference.
for define in [Python3.define, TypeScript5.define]:  # Python's and TypeScript's versions, beside their domains
    store.transaction(define)
FS = Framework.Standards
c, cpp = store.Instances.Expressions.C.Expressions, store.Instances.Expressions.Cpp.Expressions
py3, ts5 = store.Instances.Expressions.Python.Python3, store.Instances.Expressions.TypeScript.TypeScript5
assert list(FS.standards(c)) == ["ISO/IEC 9899:2011", "ISO/IEC 9899:2018", "ISO/IEC 9899:2024"]
c11 = c.standards({"designation": "ISO/IEC 9899:2011"})  # by its path: its dialect's, then its designation
assert FS.standards(c)["ISO/IEC 9899:2011"] is c11 and FS.designation_of(c11) == "ISO/IEC 9899:2011"
assert FS.tags(c11) == ("ISO/IEC 9899:2011", "C11") and FS.find(c, "C11") is FS.find(c, "ISO/IEC 9899:2011") is c11
assert FS.find(c, "C++17") is None and FS.find(cpp, "C11") is None  # each dialect's own editions
assert FS.properties(c11) == {"bodies": ("org.iso", "ch.iec"), "number": "9899", "revision": ("year", 2011),
                              "names": ("C11",)}  # a joint standard, by its bodies' reverse-domain names
assert [FS.tags(s) for s in FS.standards(c).values()] == [
    ("ISO/IEC 9899:2011", "C11"), ("ISO/IEC 9899:2018", "C17"), ("ISO/IEC 9899:2024", "C23")]
assert [FS.tags(s) for s in FS.standards(cpp).values()] == [
    ("ISO/IEC 14882:2017", "C++17"), ("ISO/IEC 14882:2020", "C++20"), ("ISO/IEC 14882:2024", "C++23")]
assert FS.properties(FS.find(cpp, "C++23"))["revision"] == ("year", 2024)  # ISO/IEC 14882:2024 is C++23
VERSIONS = tuple(f"Python 3.{minor}" for minor in range(16))  # each by its published name, with no common one
assert tuple(FS.standards(py3)) == VERSIONS and FS.tags(FS.find(py3, "Python 3.12")) == ("Python 3.12",)
assert FS.properties(FS.find(py3, "Python 3.12")) == {"bodies": ("org.python",), "revision": ("version", "3.12")}
TS_VERSIONS = tuple(f"TypeScript 5.{minor}" for minor in range(10))
assert tuple(FS.standards(ts5)) == TS_VERSIONS
assert FS.properties(FS.find(ts5, "TypeScript 5.0")) == {"bodies": ("com.microsoft",), "revision": ("version", "5.0")}
# Each kind records the editions that have it, as data: each edition its kinds, by their schemas, and a kind the
# editions of a dialect it is in; an extension and invalid syntax are in none.


def editions(dialect, singleton):
    """Each of a dialect's kinds, by its tag, and the editions that have it, by their last tags: their common names,
    or their designations where they have none."""
    union = store.registered(dialect.union)
    return {b.name: tuple(FS.tags(s)[-1] for s in FS.of(singleton, b.schema).values()) for b in union.branches}


assert C.EXTENSIONS == CPP.EXTENSIONS == {"statementexpression", "extensionexpression"}  # GNU's
READ_KINDS = {"typeid": (), "definedexpression": ()}  # C Types' and C Preprocessor's kinds, in their own dialects' editions
assert editions(C.DIALECT, c) == {**{tag: () if tag in C.EXTENSIONS | INVALID else ("C23",) if tag == "constant" else (
    "C11", "C17", "C23") for tag in C.DIALECT.kinds()}, **READ_KINDS}  # nullptr: C23's; C17 is C11's corrigenda
since_20 = {"requiresexpression", "awaitexpression", "yieldexpression", "designatedinitializer"}
assert editions(CPP.DIALECT, cpp) == {tag: () if tag in CPP.EXTENSIONS | INVALID else ("C++20", "C++23") if tag in since_20
                                      else ("C++17", "C++20", "C++23") for tag in CPP.DIALECT.kinds()} | READ_KINDS
assert TS.EXTENSIONS == {"jsxelement", "jsxfragment", "jsxemptyexpression"}  # JSX: every other kind in every version
TS_UNIONS = {d: [b.name for b in store.registered(d.union).branches] for d in TS_FAMILY}
assert editions(TS.DIALECT, ts5) == {tag: () if tag in TS.EXTENSIONS | INVALID | set(TSP.TAGS) else TS_VERSIONS
                                     for tag in TS_UNIONS[TS.DIALECT]}  # the patterns its `Term` lists, Patterns' own
tst, tsp = store.Instances.Expressions.TypeScript.Types, store.Instances.Expressions.TypeScript.Patterns
for dialect, singleton in ((TST, tst), (TSP, tsp)):  # TypeScript's types and patterns cover its versions, each its own kinds
    assert tuple(FS.standards(singleton)) == TS_VERSIONS and FS.properties(FS.find(singleton, "TypeScript 5.9")) == {
        "bodies": ("com.microsoft",), "revision": ("version", "5.9")} and dialect.SINCE == {}
    assert editions(dialect.DIALECT, singleton) == {tag: TS_VERSIONS if tag in dialect.TAGS else ()
                                                    for tag in TS_UNIONS[dialect.DIALECT]}
python = editions(P.DIALECT, py3)  # each kind from the version it arrived in
assert python[P.BINOP] == VERSIONS and python[P.NAMEDEXPR] == VERSIONS[8:] and python[P.YIELDFROM] == VERSIONS[3:]
assert python[P.TEMPLATESTR] == ("Python 3.14", "Python 3.15") and python[P.AWAIT][0] == "Python 3.5"
assert python[P.JOINEDSTR][0] == "Python 3.6"
assert P.EXTENSIONS == {"let", "import", "importfrom"} and all(python[tag] == () for tag in P.EXTENSIONS | INVALID)
pt_, pp_ = store.Instances.Expressions.Python.Types, store.Instances.Expressions.Python.Patterns  # Python's sublanguages
assert tuple(FS.standards(pt_)) == VERSIONS and tuple(FS.standards(pp_)) == VERSIONS[10:]  # its own kinds alone
assert editions(PT, pt_) == {b.name: VERSIONS[12:] if b.name in PYT.TAGS else () for b in store.registered(PT.union).branches}
assert editions(PP, pp_) == {tag: VERSIONS[10:] if tag in PYP.TAGS else () for tag in PP.kinds()}
# The regular expression dialects: Python's covers Python 3.0 to 3.15, `atomicgroup` from 3.11; TypeScript's ECMA-262's
# editions, by Ecma's designations, the v flag's kinds from ES2024, modifiers from ES2025, ES2026 adding none; C++'s
# C++'s editions, every kind in each.
pr_ = store.Instances.Expressions.Python.RegExp
assert tuple(FS.standards(pr_)) == VERSIONS and editions(PR, pr_) == {
    tag: VERSIONS[11:] if tag == "atomicgroup" else VERSIONS if tag in PYR.TAGS else () for tag in PR.kinds()}
tsr_ = store.Instances.Expressions.TypeScript.RegExp
LAST = tuple(names[-1] for _designation, _revision, names, _symbol in TSR.EDITIONS)  # ES5.1, ES2015 ... ES2026
assert tuple(FS.standards(tsr_)) == tuple(edition[0] for edition in TSR.EDITIONS) and LAST[-1] == "ES2026"
assert FS.tags(FS.find(tsr_, "ES2015")) == ("ECMA-262 6th Edition", "ECMAScript 2015", "ES2015")
assert FS.tags(FS.find(tsr_, "ES2026")) == ("ECMA-262 17th Edition", "ECMAScript 2026", "ES2026")
assert editions(TR, tsr_) == {tag: LAST[TSR.SINCE.get(tag, 0):] if tag in TSR.TAGS else () for tag in TS_UNIONS[TR]}
assert FS.kinds(FS.find(tsr_, "ES2026")) == FS.kinds(FS.find(tsr_, "ES2025"))  # its pattern grammar unchanged
xr_ = store.Instances.Expressions.Cpp.RegExp
assert [FS.tags(s) for s in FS.standards(xr_).values()] == [FS.tags(s) for s in FS.standards(cpp).values()]
assert editions(XR, xr_) == {tag: ("C++17", "C++20", "C++23") if tag in CppRegExp.Expressions.TAGS else () for tag in XR.kinds()}
# POSIX's two cover POSIX.1's editions, by IEEE's designations, the ISO/IEC designations C++ cites among their common
# names, every kind in each.
for dialect, package in zip(POSIX, (PosixBRE, PosixERE)):
    singleton = store.singleton(f"Expressions.{dialect.path}")
    assert tuple(FS.standards(singleton)) == tuple(f"IEEE Std 1003.1-{year}" for year in (2001, 2008, 2017, 2024))
    assert editions(dialect, singleton) == {tag: ("ISO/IEC 9945:2003", "ISO/IEC/IEEE 9945:2009", "POSIX.1-2017", "POSIX.1-2024")
                                            if tag in package.Expressions.TAGS else () for tag in dialect.kinds()}
assert [s for s in FS.kinds(FS.find(c, "C23"))] == [b.schema for b in C_UNION.branches if b.name not in C.EXTENSIONS | INVALID | set(READ_KINDS)]
# A kind's schema reads the same entries from its side, through its extension adjacency `standards`.
namedexpr = {b.name: b.schema for b in store.registered(P.DIALECT.union).branches}[P.NAMEDEXPR]
assert [FS.designation_of(e.standard) for e in Builders.OfObject.Builder(namedexpr, False, store).standards] == list(VERSIONS[8:])
# SystemVerilog's six dialects cover the IEEE's editions, each with its own kinds: Expressions, Types and the Preprocessor
# Verilog's (IEEE 1364) and SystemVerilog's (IEEE 1800, which merged it), Patterns, Assertions and Constraints IEEE
# 1800's alone; each kind from the edition it arrived in, as mbse-programs' SINCE has it.
VERILOG = tuple(f"IEEE Std 1364-{year}" for year in (1995, 2001, 2005))
IEEE1800 = tuple(f"IEEE Std 1800-{year}" for year in (2005, 2009, 2012, 2017, 2023))
LABELS = (*(f"Verilog-{year}" for year in (1995, 2001, 2005)), *(f"SystemVerilog-{year}" for year in (2005, 2009, 2012, 2017, 2023)))
assert [e[0] for e in SV.EDITIONS] == [*VERILOG, *IEEE1800] and [e[3] for e in SV.EDITIONS] == list(LABELS)
singletons = {dialect: store.singleton(f"Expressions.{dialect.path}") for dialect in FAMILY}
for dialect, singleton in singletons.items():
    whole = dialect in (SV.DIALECT, SVT.DIALECT, SVPP.DIALECT)
    assert tuple(FS.standards(singleton)) == ((*VERILOG, *IEEE1800) if whole else IEEE1800), dialect.name()
sv_ = singletons[SV.DIALECT]
assert FS.tags(FS.find(sv_, "SystemVerilog-2023")) == ("IEEE Std 1800-2023", "SystemVerilog-2023")
assert FS.properties(FS.find(sv_, "Verilog-2005")) == {"bodies": ("org.ieee",), "number": "1364", "revision": ("year", 2005),
                                                      "names": ("Verilog-2005",)}  # Verilog, IEEE 1364, its subset
for dialect, own in [(SV.DIALECT, SV.TAGS), (SVT.DIALECT, SVT.TAGS), (SVP.DIALECT, SVP.TAGS), (SVA.DIALECT, SVA.TAGS),
                     (SVC.DIALECT, SVC.TAGS), (SVPP.DIALECT, SVPP.TAGS)]:  # each edition its own kinds alone
    whole = dialect in (SV.DIALECT, SVT.DIALECT, SVPP.DIALECT)
    since = {**SV.SINCE, **SVT.SINCE, **SVA.SINCE, **SVC.SINCE}  # a kind not listed is in every edition its dialect covers; the dialects it reads,
    assert editions(dialect, singletons[dialect]) == {
        b.name: LABELS[since.get(b.name, 0 if whole else 3):] if b.name in own else ()
        for b in store.registered(dialect.union).branches}, dialect.name()  # in theirs
assert editions(SV.DIALECT, sv_)["castexpression"] == LABELS[3:] and editions(SVT.DIALECT, singletons[SVT.DIALECT])["structtype"] == LABELS[3:]
assert editions(SVPP.DIALECT, singletons[SVPP.DIALECT])["macrousage"] == LABELS
assert store.transaction(SVTypes.define)[0]["SystemVerilog_2023"] is FS.find(singletons[SVT.DIALECT], "SystemVerilog-2023")
# Every other dialect covers none: its code names no standard.
assert all(FS.standards(store.singleton(f"Expressions.{path}")) == {} for path in [
    "Patterns.Expressions", "MATLAB.Expressions", "Excel.Expressions", "LaTeX.Expressions"])
# Run again, a define writes nothing; an edition validates. The model: an edition a reference object its dialect owns
# by designation, its revision a year, an edition or a version, and a set relating editions to kinds' schemas.
assert store.transaction(C.define)[0]["C11"] is c11 and store.transaction(Cpp.define)[0]["Cpp17"] is FS.find(cpp, "C++17")
Standard = store.registered("Expressions.Framework.Standard")
assert Validators.Validate(store)(Standard, c11) == []
assert Standard.ref and list(Standard.properties) == ["bodies", "number", "revision", "names"]
assert list(Standard.adjacencies) == ["dialect", "kinds"] and Standard.adjacencies["dialect"].owner
assert [b.name for b in store.registered("Expressions.Framework.Revision").branches] == ["year", "edition", "version"]
Covers, HasStandard = store.registered("Expressions.Framework.Covers"), store.registered("Expressions.Framework.HasStandard")
assert Covers.uniques == (frozenset({"dialect", "designation"}), frozenset({"standard"}))
assert list(HasStandard.entry.links) == ["kind", "standard"] and HasStandard.uniques == (frozenset(),)  # a set
assert "standards" in store.registered("Schemas.OfObject").adjacencies
# Until a dialect covers it, an edition has no designation, and is found by its common names alone.
with store.transaction():
    loose = store.Expressions.Framework.Standard(lambda s: s.names(["loose"]))
    assert FS.designation_of(loose) is None and FS.tags(loose) == ("loose",)
    store.Expressions.Patterns.Expressions.Dialect(store.singleton("Expressions.Patterns.Expressions"),
                                                   lambda b: b.standards({"Loose 1": lambda e: e.standard(loose)}))
assert FS.tags(loose) == ("Loose 1", "loose") and FS.find(store.singleton("Expressions.Patterns.Expressions"), "loose") is loose